# دمج LoRA مع Gemma 4 12B — النسخة المصححة (يحافظ على الوسائط المتعددة)

**المشاكل اللي انصلحت من الدفتر الأصلي:**

| # | المشكلة | الأثر | الحل |
|---|---|---|---|
| 1 | التحميل بـ `AutoModelForCausalLM` | يحمّل الجزء اللغوي فقط — أوزان الرؤية/الصوت تنحذف من النموذج المحفوظ | `AutoModelForImageTextToText` |
| 2 | حفظ `AutoTokenizer` فقط | ضياع `preprocessor_config.json` و `processor_config.json` → النموذج ما يستقبل صور/صوت حتى لو الأوزان سليمة | حفظ `AutoProcessor` (يشمل الـ tokenizer + كل ملفات المعالجة) |
| 3 | خلية تحميل أولى بدون `torch_dtype` | تحميل بـ float32 = ضعف الذاكرة + خطر OOM على A100 40GB | خلية محذوفة (مكررة أصلاً) |
| 4 | `hf_hub_download` بدون token بخلية التحقق (الخطوة 7) | Gemma مستودع مقيّد (gated) → فشل صامت بجلب ملفات المعالج | تمرير الـ token |
| 5 | فحص المفاتيح يقارن مع base محمّل بكلاس ناقص | التحقق يعطي ✅ كاذب لأن المرجع نفسه ناقص | المرجع الآن الكلاس الكامل + فحص صريح لوجود مفاتيح `vision_tower` و `audio_tower` |
| 6 | `push_to_hub` للـ tokenizer بس | نفس مشكلة رقم 2 على الريبو | رفع مجلد كامل يحتوي ملفات الـ processor |

**القاعدة الذهبية:** الـ LoRA عندك مقيّد بالـ regex على طبقات اللغة فقط، فالدمج ما يمس أبراج الرؤية/الصوت — الخطر كله بمرحلة **التحميل والحفظ**، مو بالأوزان.

In [1]:
# 1) تسجيل الدخول
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('ameerwisam2005')
login(token=hf_token)

In [2]:
# 2) الدمج الصحيح — بالكلاس الكامل متعدد الوسائط
!pip install --upgrade torchao

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor  # ✅ مو AutoModelForCausalLM
from peft import PeftModel

base_model_id = "google/gemma-4-12B-it"
adapter_model_id = "ameer4wisam/gemma-iraqi-finetune"
save_dir = "./gemma-iraqi-merged"

# ✅ الكلاس الكامل يحمّل نموذج اللغة + برج الرؤية + برج الصوت
base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    dtype=torch.bfloat16,   # ✅ تم تصحيح torch_dtype إلى dtype
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_model_id)
merged_model = model.merge_and_unload()

merged_model.save_pretrained(save_dir, safe_serialization=True)

# ✅ AutoProcessor يحفظ: tokenizer + preprocessor_config.json + processor_config.json
#    + special_tokens_map.json + chat_template — كلها بأمر واحد
processor = AutoProcessor.from_pretrained(base_model_id)
processor.save_pretrained(save_dir)

print("✅ تم الدمج والحفظ بالكلاس الكامل")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

✅ تم الدمج والحفظ بالكلاس الكامل


In [3]:
import torch
from transformers import AutoModelForImageTextToText, AutoTokenizer

model_check = AutoModelForImageTextToText.from_pretrained(
    save_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model_check.eval()

enc = model_check.config  # النموذج محمّل، جرّب توليد حتمي
tok = AutoTokenizer.from_pretrained(save_dir)
e = tok.apply_chat_template(
    [{"role": "user", "content": "شكد سعر المكيف؟"}],
    add_generation_prompt=True, return_tensors="pt", return_dict=True)
out = model_check.generate(
    input_ids=e["input_ids"].to(model_check.device),
    attention_mask=e["attention_mask"].to(model_check.device),
    max_new_tokens=64, do_sample=False)
print(tok.decode(out[0][e["input_ids"].shape[1]:], skip_special_tokens=True))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

سعر المكيف يوصل لـ700,000 دينار، وإذا تريد أحسن شي بالسعر أنطيك ثنين وسعر خاص.


In [4]:
# المقارنة الصحيحة: شنو عند Google وشنو عندك
import os
from huggingface_hub import list_repo_files, hf_hub_download
import shutil

base_repo = "google/gemma-4-12B-it"
remote_files = list_repo_files(base_repo, token=hf_token)

# الملفات غير الوزنية اللي تهمنا (إعدادات + معالج + توكنايزر)
config_files = [f for f in remote_files
                if f.endswith((".json", ".jinja", ".model", ".txt"))
                and not f.startswith(".")
                and "index" not in f]          # index الأوزان ماله علاقة، أوزانك مختلفة

print("ملفات الإعدادات بالريبو الأساسي:")
missing = []
for f in config_files:
    exists = os.path.exists(os.path.join(save_dir, f))
    print(("✅" if exists else "❌"), f)
    if not exists:
        missing.append(f)

# نسخ الناقص فقط
for f in missing:
    p = hf_hub_download(repo_id=base_repo, filename=f, token=hf_token)
    shutil.copy(p, os.path.join(save_dir, f))
    print(f"📥 نُسخ: {f}")

print("\n✅ مجلدك هسه مطابق للريبو الأساسي بملفات الإعدادات")

ملفات الإعدادات بالريبو الأساسي:
✅ chat_template.jinja
✅ config.json
✅ generation_config.json
✅ processor_config.json
✅ tokenizer.json
✅ tokenizer_config.json

✅ مجلدك هسه مطابق للريبو الأساسي بملفات الإعدادات


In [5]:
from transformers import AutoProcessor
proc = AutoProcessor.from_pretrained(save_dir)
print(type(proc).__name__)
print("image_processor:", type(getattr(proc, "image_processor", None)).__name__)

Gemma4UnifiedProcessor
image_processor: Gemma4UnifiedImageProcessor


In [6]:
# 1) فحص الريبو المرفوع (ثواني، بدون أوزان)
from transformers import AutoConfig, AutoProcessor

repo = "ameer4wisam/gemma-iraqi-finetune-v2"
cfg = AutoConfig.from_pretrained(repo)
proc = AutoProcessor.from_pretrained(repo)

print("الإعدادات:", type(cfg).__name__)
print("المعالج:", type(proc).__name__)
print("معالج الصور:", type(getattr(proc, "image_processor", None)).__name__)

config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.75k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

الإعدادات: Gemma4UnifiedConfig
المعالج: Gemma4UnifiedProcessor
معالج الصور: Gemma4UnifiedImageProcessor


In [7]:
# 2) الدليل النهائي: استدلال بصورة من الريبو مباشرة
import torch, requests
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(repo)
model = AutoModelForImageTextToText.from_pretrained(
    repo, torch_dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager").eval()

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "شنو تشوف بالصورة؟"}]}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt")
inputs = {k: v.to(model.device, dtype=torch.bfloat16) if v.is_floating_point() else v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

هذي صورة لقطة.


In [8]:
# 4) الرفع المباشر (بدون شروط أو تحقق)
import os
os.environ["HF_HUB_USE_XET"] = "0"  # تجاوز مشاكل Xet التجريبي
import huggingface_hub._commit_api
huggingface_hub._commit_api.is_xet_available = lambda: False

from huggingface_hub import HfApi

target_repo = "ameer4wisam/gemma-iraqi-finetune-v2"
api = HfApi(token=hf_token)
api.create_repo(repo_id=target_repo, exist_ok=True)

# رفع المجلد بالكامل
api.upload_folder(
    folder_path=save_dir,
    repo_id=target_repo,
    repo_type="model",
    commit_message="Forced Upload: Merged LoRA with model weights and files",
)
print(f"🎉 تم الرفع إلى {target_repo}")

🎉 تم الرفع إلى ameer4wisam/gemma-iraqi-finetune-v2


### 5) فحص ما بعد الرفع — تأكد الوسائط رجعت

هذا الفحص يحسم إذا المشكلة انحلت: نحمّل من الريبو مباشرة ونتأكد من نوع المعالج ووجود إعدادات الرؤية.

In [9]:
# 5) فحص سريع من الريبو (بدون تحميل الأوزان — ثواني فقط)
from transformers import AutoProcessor, AutoConfig

target_repo = "ameer4wisam/gemma-iraqi-finetune-v2"

cfg = AutoConfig.from_pretrained(target_repo)
proc = AutoProcessor.from_pretrained(target_repo)

print(f"نوع الإعدادات: {type(cfg).__name__}")
print(f"نوع المعالج:  {type(proc).__name__}")

checks = {
    "vision_config موجود": hasattr(cfg, "vision_config") and cfg.vision_config is not None,
    "المعالج كامل (مو مجرد tokenizer)": "Processor" in type(proc).__name__,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

# audio_config معلوماتي فقط: نسخة 12B قد تكون رؤية بس (بعكس E4B اللي بيها برج صوت)
has_audio = hasattr(cfg, "audio_config") and cfg.audio_config is not None
print(("✅" if has_audio else "ℹ️"), "audio_config", "موجود" if has_audio else "غير موجود (طبيعي إذا النسخة رؤية فقط)")

if all(checks.values()):
    print("\n🎉 الوسائط المتعددة سليمة بالنموذج المرفوع")
else:
    print("\n🛑 بعدها ناقصة — أعد خلية الدمج (2) والتحقق (3)")

نوع الإعدادات: Gemma4UnifiedConfig
نوع المعالج:  Gemma4UnifiedProcessor
✅ vision_config موجود
✅ المعالج كامل (مو مجرد tokenizer)
✅ audio_config موجود

🎉 الوسائط المتعددة سليمة بالنموذج المرفوع


In [10]:
# 6) اختبار استدلال نصي (وصفة الاستدلال المعتمدة: حتمي، eager، 64 توكن)
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "ameer4wisam/gemma-iraqi-finetune-v2"

processor = AutoProcessor.from_pretrained(model_id)
tokenizer = processor.tokenizer
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",   # مطلوب لمعمارية Gemma 4
).eval()

# اكتشاف توكن التوقف تلقائياً (لا تكتبه يدوياً)
_probe_enc = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt")
_probe = _probe_enc["input_ids"][0].tolist()

special = set(tokenizer.all_special_ids)
stop_ids = list({int(t) for t in _probe[-3:] if int(t) in special}
                | {tokenizer.eos_token_id})

messages = [
    {"role": "system", "content": "أنت بائع عراقي محترف بمحل أجهزة كهربائية. جاوب باللهجة العراقية، قصير ومباشر."},
    {"role": "user", "content": "هلو، عندكم مكيفات سبلت؟"},
]
enc = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True)
with torch.no_grad():
    out = model.generate(
        input_ids=enc["input_ids"].to(model.device),
        attention_mask=enc["attention_mask"].to(model.device),
        eos_token_id=stop_ids, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

إي أكو، سبلت 1 طن، انفرتر، ضمان سنتين. بـ725,000 دينار


In [11]:
# 7) اختبار وسائط (صورة) — الدليل النهائي إن الوسائط تشتغل
import requests
from PIL import Image

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "شنو تشوف بالصورة؟"},
    ],
}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

قطة برية تمشي على الثلج.


In [12]:
# ============================================================
#  خلية الاستدلال الاحترافية — gemma-iraqi-finetune-v2
#  الوصفة المعتمدة: حتمي | eager | 64 توكن | كتالوج محقون | فحص أرقام
# ============================================================
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"   # أو "./gemma-iraqi-merged" محلياً

# ---------- 1) التحميل (كلاس كامل متعدد الوسائط + تقرير تحميل) ----------
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

model, _info = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",        # إلزامي لمعمارية Gemma 4
    output_loading_info=True,           # فحص السلامة المعتمد
)
model.eval()
assert not _info["missing_keys"],    f"🛑 أوزان ناقصة: {_info['missing_keys'][:5]}"
assert not _info["unexpected_keys"], f"🛑 أوزان غريبة (بقايا LoRA?): {_info['unexpected_keys'][:5]}"
print(f"✅ النموذج سليم | المعالج: {type(processor).__name__}")

# ---------- 2) اكتشاف توكنات التوقف تلقائياً (لا تكتبها يدوياً) ----------
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt",
)["input_ids"][0].tolist()
_special = set(tokenizer.all_special_ids)
STOP_IDS = list({t for t in _probe[-3:] if t in _special} | {tokenizer.eos_token_id})
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else STOP_IDS[0]
print(f"✅ توكنات التوقف: {STOP_IDS}")

# ---------- 3) الكتالوج + System Prompt (الحقائق هنا، مو بالأوزان) ----------
CATALOG = """المنتجات المتوفرة حالياً (التزم بيها حرفياً):

الماركات المتوفرة: LG فقط (إذا سأل الزبون عن الماركات، گله عدنا LG بس هسه)

- مكيف LG سبلت طن واحد: 700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة
- مكيف LG سبلت طن ونص: 950,000 دينار، ضمان سنتين، استهلاك ~1.5 كيلوواط/ساعة
  (الفرق بالكهرباء: الطن ونص يصرف حوالي 50% أكثر من الطن الواحد)

- التركيب: 50,000 دينار للجهاز الواحد
- التوصيل داخل بغداد: مجاني | خارج بغداد: 20,000 دينار

خصم شراء قطعتين 5% (المجاميع محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد: 1,400,000 قبل الخصم -> 1,330,000 بعد الخصم
- 2 × طن ونص: 1,900,000 قبل الخصم -> 1,805,000 بعد الخصم"""

SYSTEM_PROMPT = f"""أنت بائع عراقي محترف بمحل أجهزة كهربائية.

{CATALOG}

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم ولا تحسب مجاميع جديدة.
4. جاوب على السؤال المطلوب بالضبط - لا تكرر معلومات ما انسألت عنها.
5. إذا الزبون گال يريد "اثنين" بدون تحديد الموديل، اسأله: "اثنين طن واحد لو طن ونص؟"
6. لا تدّعي مخزون أو ندرة أو عروض غير مكتوبة بالقائمة.
7. إذا طلب منتج غير موجود، گله: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
8. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
9. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب."""

# ---------- 4) حارس الأرقام (regression guard) ----------
_ALLOWED = set(re.findall(r"\d[\d,\.]*", CATALOG))

def check_numbers(reply: str) -> list:
    """أي رقم مالي بالرد لازم يكون موجود حرفياً بالكتالوج."""
    bad = []
    for num in re.findall(r"\d[\d,\.]*", reply):
        clean = num.rstrip(".,")
        if clean in _ALLOWED:
            continue
        if "," not in clean and "." not in clean and len(clean) <= 2:
            continue  # عدد قطع/سنين، مو سعر
        bad.append(clean)
    return bad

# ---------- 5) دالة الاستدلال الموحدة (نص + صورة اختيارية) ----------
class IraqiSalesBot:
    """محادثة متعددة الأدوار: حتمي دائماً، كتالوج محقون، فحص أرقام لكل رد."""

    def __init__(self, system_prompt: str = SYSTEM_PROMPT):
        self.messages = [{"role": "system", "content": system_prompt}] if system_prompt else []

    def _generate(self, inputs) -> str:
        with torch.no_grad():
            out = model.generate(
                **inputs,
                eos_token_id=STOP_IDS,
                pad_token_id=PAD_ID,
                max_new_tokens=64,       # أجوبة التدريب قصيرة؛ الطول الزائد = هذيان
                do_sample=False,          # حتمي فقط — الوضع المتوازن محذوف نهائياً
            )
        return tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

    def chat(self, user_text: str, image=None, verbose: bool = True) -> str:
        # بناء محتوى الرسالة (نص، أو نص + صورة)
        if image is not None:
            content = [{"type": "image", "image": image},
                       {"type": "text", "text": user_text}]
        else:
            content = user_text
        self.messages.append({"role": "user", "content": content})

        # الترميز: processor للوسائط، tokenizer للنص الصرف
        if image is not None:
            inputs = processor.apply_chat_template(
                self.messages, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt",
            ).to(model.device, dtype=torch.bfloat16)
        else:
            enc = tokenizer.apply_chat_template(
                self.messages, add_generation_prompt=True,
                return_dict=True, return_tensors="pt",
            )
            inputs = {k: v.to(model.device) for k, v in enc.items()}

        reply = self._generate(inputs)

        # نسجل بالسجل نسخة نصية (حتى ما نعيد إرسال الصورة بكل دور)
        self.messages[-1] = {"role": "user", "content": user_text}
        self.messages.append({"role": "assistant", "content": reply})

        # حارس الأرقام
        bad = check_numbers(reply)
        if verbose:
            ctx = len(tokenizer.apply_chat_template(self.messages, add_generation_prompt=False))
            print(f"👤 {user_text}")
            print(f"🤖 {reply}   (سياق: {ctx} توكن)")
            if bad:
                print(f"   🚨 أرقام مو بالكتالوج: {bad}")
            print("-" * 60)
        return reply

    def reset(self):
        self.messages = self.messages[:1]  # يحتفظ بالـ system prompt فقط

# ---------- 6) الاستخدام ----------
bot = IraqiSalesBot()
bot.chat("هلو، عندكم مكيفات سبلت؟")
bot.chat("شكد سعر الطن الواحد؟")
bot.chat("وإذا أخذت اثنين طن ونص؟")

# مثال ويا صورة (اختياري):
# from PIL import Image
# img = Image.open("device.jpg")
# bot.chat("عندكم مثل هذا المكيف؟", image=img)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

✅ النموذج سليم | المعالج: Gemma4UnifiedProcessor
✅ توكنات التوقف: [1, 106]
👤 هلو، عندكم مكيفات سبلت؟
🤖 عدنا طن واحد بـ700,000 أو طن ونص بـ950,000، شتفضل؟   (سياق: 2 توكن)
------------------------------------------------------------
👤 شكد سعر الطن الواحد؟
🤖 مكيف LG سبلت طن واحد بـ700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة   (سياق: 2 توكن)
------------------------------------------------------------
👤 وإذا أخذت اثنين طن ونص؟
🤖 إي، اثنين طن ونص يطلعلك 1,805,000 بعد خصم 5%   (سياق: 2 توكن)
------------------------------------------------------------


'إي، اثنين طن ونص يطلعلك 1,805,000 بعد خصم 5%'

In [13]:
# ============================================================
#  الاستدلال النهائي — Gemma Iraqi v2 (دروع ثنائية الاتجاه)
#
#  فلسفة الدرع الثنائي:
#    الدرع يُبنى من الكتالوج نفسه وقت التشغيل — مو قائمة ثابتة:
#    • الموضوع موجود بالكتالوج (مثل الضمان هنا) → الرد مسموح،
#      ويا فحص مطابقة: المدة/الرقم لازم يطابق الكتالوج حرفياً.
#    • الموضوع غير موجود → الرد لازم يحيل ("اتأكدلك") وإلا يُستبدل.
#    تغيّر الكتالوج؟ الدرع يتكيف تلقائياً بدون تعديل كود.
#
#  القواعد الحديدية المطبقة:
#    توقف مكتشف تلقائياً (probe + generation_config + حصانة <turn|>)
#    حتمي للحقائق | بدون repetition_penalty | eager | الحقائق بالكتالوج
# ============================================================
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"   # أو "./gemma-iraqi-merged" محلياً

# ============================================================
# 1) التحميل (وصفتك المعتمدة: كلاس كامل + تقرير تحميل)
# ============================================================
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

model, _info = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    output_loading_info=True,
)
model.eval()
assert not _info["missing_keys"],    f"🛑 أوزان ناقصة: {_info['missing_keys'][:5]}"
assert not _info["unexpected_keys"], f"🛑 أوزان غريبة (بقايا LoRA?): {_info['unexpected_keys'][:5]}"
print(f"✅ النموذج سليم | المعالج: {type(processor).__name__}")

# ============================================================
# 2) التوقف: ثلاث طبقات اكتشاف (لا hardcode أبداً)
# ============================================================
# أ) من generation_config
_gc = model.generation_config.eos_token_id
STOP_IDS = list(_gc) if isinstance(_gc, (list, tuple)) else ([_gc] if _gc is not None else [])
# ب) من probe على القالب (آخر توكنات دور مغلق)
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"}, {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt",
)["input_ids"][0].tolist()
_special = set(tokenizer.all_special_ids)
STOP_IDS += [t for t in _probe[-3:] if t in _special and t not in STOP_IDS]
# ج) حصانة توكنات إنهاء الدور المعروفة (الـ trainer/الدمج ممكن يمسحها)
for tok in ("<turn|>", "<end_of_turn>"):
    tid = tokenizer.convert_tokens_to_ids(tok)
    if tid is not None and tid != tokenizer.unk_token_id and tid not in STOP_IDS:
        STOP_IDS.append(int(tid))
if tokenizer.eos_token_id is not None and tokenizer.eos_token_id not in STOP_IDS:
    STOP_IDS.append(tokenizer.eos_token_id)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else STOP_IDS[0]
assert len(STOP_IDS) >= 2, "🛑 توكن إنهاء الدور مفقود — التوليد سيهذي!"
print(f"✅ توكنات التوقف: {STOP_IDS} -> {[repr(tokenizer.decode([t])) for t in STOP_IDS]}")

THINKING_KWARG = next((c for c in ("enable_thinking", "thinking", "add_thinking")
                       if c in (tokenizer.chat_template or "")), None)
print(f"✅ thinking kwarg: {THINKING_KWARG or 'غير موجود'}")

# ============================================================
# 3) الكتالوج — مصدر الحقيقة الوحيد (الضمان موجود هنا عمداً)
# ============================================================
CATALOG = """المنتجات المتوفرة حالياً (التزم بيها حرفياً):

الماركات المتوفرة: LG فقط (إذا سأل الزبون عن الماركات، گله عدنا LG بس هسه)

- مكيف LG سبلت طن واحد: 700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة
- مكيف LG سبلت طن ونص: 950,000 دينار، ضمان سنتين، استهلاك ~1.5 كيلوواط/ساعة
  (الفرق بالكهرباء: الطن ونص يصرف حوالي 50% أكثر من الطن الواحد)

- التركيب: 50,000 دينار للجهاز الواحد
- التوصيل داخل بغداد: مجاني | خارج بغداد: 20,000 دينار

خصم شراء قطعتين 5% (المجاميع محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد: 1,400,000 قبل الخصم -> 1,330,000 بعد الخصم
- 2 × طن ونص: 1,900,000 قبل الخصم -> 1,805,000 بعد الخصم"""

SYSTEM_PROMPT = f"""أنت بائع عراقي محترف بمحل أجهزة كهربائية.

{CATALOG}

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم ولا تحسب مجاميع جديدة.
4. جاوب على السؤال المطلوب بالضبط - لا تكرر معلومات ما انسألت عنها.
5. إذا الزبون طلب شراء "اثنين" او كمية بدون تحديد الموديل، اسأله: "اثنين طن واحد لو طن ونص؟". لا تسأل هذا السؤال إذا الزبون حدد الموديل، ولا تسأله بالتحيات، ولا تكرره.
6. لا تدّعي مخزون أو ندرة أو عروض غير مكتوبة بالقائمة.
7. إذا طلب منتج غير موجود، گله: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
8. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
9. إذا ما تعرف معلومة أو مو مكتوبة بالقائمة، گول "أتأكدلك وأرد عليك" ولا تخترع جواب."""

# ============================================================
# 4) الدروع ثنائية الاتجاه — تُبنى من الكتالوج ديناميكياً
# ============================================================
# مجموعات المواضيع الحساسة: (اسم، كلمات السؤال، كلمات وجودها بالكتالوج)
TOPIC_GROUPS = {
    "ضمان":    ["ضمان", "كفالة", "گارنتي", "وارنتي", "الكفالة"],
    "توصيل":   ["توصيل", "شحن", "يوصلون"],
    "تركيب":   ["تركيب", "نصب", "التنصيب"],
    "استهلاك": ["استهلاك", "كهرباء", "كيلوواط", "امبير", "أمبير"],
    "لون":     ["لون", "الوان", "ألوان"],
    "برنامج":  ["برنامج", "بروگرام", "بروجرام"],
    "منشأ":    ["منشأ", "صناعة", "المنشأ"],
    "تقسيط":   ["تقسيط", "اقساط", "أقساط", "قسط"],
    "صيانة":   ["صيانة", "تصليح", "قطع غيار"],
}

DURATION_WORDS = ["سنة", "سنتين", "سنوات", "شهر", "شهرين", "أشهر", "اشهر", "شهور"]
# ادعاءات تغطية/استثناء داخل موضوع الضمان — مسموحة فقط اذا مكتوبة حرفياً بالكتالوج
COVERAGE_CLAIMS = ["يشمل", "ما يشمل", "شامل", "يغطي", "ما يغطي", "الكسر",
                   "الماي", "الماء", "كومبريسر", "الكومبريسر", "الضاغط",
                   "قطع الغيار", "الصيانة المجانية", "استبدال"]
CONFIRM_PHRASES = ["اتأكدلك", "أتأكدلك", "أتأكد لك"]
SAFE_REPLY = "والله ما أگدر أجاوبك بالضبط هسه، أتأكدلك وأرد عليك"
ARABIC_PREFIX = re.compile(r'^(?:وال|بال|لل|فال|ال|و|ب|ف)')


def _word_set(text: str) -> set:
    """كلمات عربية مطابقة على مستوى الكلمة (محصّن ضد 'شلونك' ≠ 'لون')"""
    words = re.findall(r'[\u0600-\u06FF]+', text)
    return {ARABIC_PREFIX.sub('', w) for w in words} | set(words)


def build_guards(catalog_text: str):
    """
    يبني الدرعين من نص الكتالوج المعطى:
      numbers_guard(reply, user_msg)  -> قائمة أرقام مخالفة
      topic_guard(reply, user_msg)    -> (الرد النهائي، سبب التدخل أو None)
    الاتجاهان:
      • موضوع بالكتالوج  -> مسموح + فحص مطابقة الحقيقة (مدد الضمان مثلاً)
      • موضوع مو بالكتالوج -> إحالة إلزامية
    """
    catalog_words = _word_set(catalog_text)

    # الاتجاه الأول: أي مواضيع "مرخّصة" لأنها موجودة بالكتالوج؟
    allowed_topics = {name for name, kws in TOPIC_GROUPS.items()
                      if any(k in catalog_words or k in catalog_text for k in kws)}

    # حقائق الضمان المعتمدة (لو موجود): المدد المذكورة حصراً
    warranty_facts = set()
    if "ضمان" in allowed_topics:
        for m in re.finditer(r'ضمان\s+([\u0600-\u06FF]+)', catalog_text):
            warranty_facts.add(m.group(1))
        warranty_facts |= {w for w in DURATION_WORDS if re.search(rf'ضمان\s+\S*\s*{w}', catalog_text)}

    # الأرقام المسموحة: كل رقم مكتوب بالكتالوج (يشمل المجاميع المحسوبة مسبقاً)
    allowed_numbers = set(re.findall(r'\d[\d,\.]*', catalog_text))

    def numbers_guard(reply: str, user_msg: str = "", extra_allowed=()) -> list:
        text = re.sub(r'(\d+)\s*(?:ألف|الف)', lambda m: f"{m.group(1)},000", reply)
        user_nums = set(re.findall(r'\d[\d,\.]*', user_msg))
        bad = []
        for num in re.findall(r'\d[\d,\.]*', text):
            clean = num.rstrip('.,')
            if clean in allowed_numbers or clean in user_nums or clean in extra_allowed:
                continue
            if ',' not in clean and '.' not in clean and len(clean) <= 2:
                continue  # عدد قطع/سنين — مو سعر
            bad.append(clean)
        return bad

    def topic_guard(reply: str, user_msg: str):
        if user_msg.startswith("[النظام]"):
            return reply, None
        user_words = _word_set(user_msg)
        reply_words = _word_set(reply)

        for name, kws in TOPIC_GROUPS.items():
            asked = any(k in user_words for k in kws)
            mentioned = any(k in reply_words for k in kws)
            if not (asked or mentioned):
                continue

            if name not in allowed_topics:
                # ⛔ الاتجاه الثاني: الموضوع مو بالكتالوج -> إحالة إلزامية
                if not any(p in reply for p in CONFIRM_PHRASES):
                    return SAFE_REPLY, f"'{name}' غير موجود بالكتالوج — منع هلوسة"
            else:
                # ✅ الاتجاه الأول: مرخّص — بس الترخيص بحدود النص المكتوب حرفياً
                if name == "ضمان" and mentioned and warranty_facts:
                    claimed = [w for w in DURATION_WORDS if w in reply_words]
                    wrong = [w for w in claimed if w not in warranty_facts]
                    # التجول داخل الموضوع المرخّص: تفاصيل تغطية غير مكتوبة بالكتالوج
                    fabricated = [c for c in COVERAGE_CLAIMS
                                  if c in reply and c not in catalog_text]
                    if wrong or fabricated:
                        return SAFE_REPLY, f"تفاصيل ضمان غير مكتوبة بالكتالوج: {wrong + fabricated}"
        return reply, None

    return numbers_guard, topic_guard, allowed_topics


numbers_guard, topic_guard, ALLOWED_TOPICS = build_guards(CATALOG)
print(f"🛡️ مواضيع مرخّصة من الكتالوج: {sorted(ALLOWED_TOPICS)}")
print(f"🛡️ مواضيع محجوبة (إحالة إلزامية): {sorted(set(TOPIC_GROUPS) - ALLOWED_TOPICS)}")

# ============================================================
# 5) البوت
# ============================================================
GEN_FACTUAL = dict(max_new_tokens=96, do_sample=False)   # 64 كانت تگص ردود 12B


class IraqiSalesBot:
    def __init__(self, system_prompt: str = SYSTEM_PROMPT):
        self.messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
        self.guard_log = []   # (رسالة الزبون، الرد الأصلي، السبب)

    def _generate(self, inputs) -> str:
        with torch.no_grad():
            out = model.generate(**inputs, eos_token_id=STOP_IDS,
                                 pad_token_id=PAD_ID, **GEN_FACTUAL)
        key = "input_ids"
        reply = tokenizer.decode(out[0][inputs[key].shape[1]:],
                                 skip_special_tokens=True).strip()
        reply = re.split(r'</?thought>|<turn\|>|\n?(?:user|model)\b', reply)[0].strip()
        half = len(reply) // 2
        if half > 20 and reply[:half].strip() == reply[half:half * 2].strip():
            reply = reply[:half].strip()
        return reply

    def chat(self, user_text: str, image=None, verbose: bool = True) -> str:
        content = ([{"type": "image", "image": image},
                    {"type": "text", "text": user_text}] if image is not None else user_text)
        self.messages.append({"role": "user", "content": content})

        kw = dict(add_generation_prompt=True, return_dict=True, return_tensors="pt")
        if THINKING_KWARG:
            kw[THINKING_KWARG] = False
        if image is not None:
            inputs = processor.apply_chat_template(self.messages, tokenize=True, **kw
                                                   ).to(model.device, dtype=torch.bfloat16)
        else:
            enc = tokenizer.apply_chat_template(self.messages, **kw)
            inputs = {k: v.to(model.device) for k, v in enc.items()}

        raw = self._generate(inputs)

        # 🛡️ الدرعان بالتسلسل: المواضيع أولاً (يفهم السياق)، الأرقام ثانياً
        reply, reason = topic_guard(raw, user_text)
        if reason is None:
            bad = numbers_guard(reply, user_text)
            if bad:
                reply, reason = SAFE_REPLY, f"أرقام مخترعة: {bad}"
        if reason:
            self.guard_log.append((user_text, raw, reason))

        self.messages[-1] = {"role": "user", "content": user_text}
        self.messages.append({"role": "assistant", "content": reply})

        if verbose:
            print(f"👤 {user_text}")
            print(f"🤖 {reply}")
            if reason:
                print(f"   🛡️ الدرع تدخّل: {reason}")
                print(f"   (الرد الأصلي: {raw[:90]})")
            print("-" * 60)
        return reply

    def reset(self):
        self.messages = self.messages[:1]


# ============================================================
# 6) اختبارات وحدة للدروع نفسها (بدون موديل — تثبت الاتجاهين)
# ============================================================
print("\n" + " 🔬 اختبارات وحدة الدروع (الاتجاهان) ".center(62, "="))

UNIT = []
def unit(name, ok):
    UNIT.append((name, bool(ok)))
    print(f"   {'✅' if ok else '❌'} {name}")

# كتالوج بديل بلا ضمان ولا توصيل — لإثبات الاتجاه الثاني بنفس الكود
_ng2, _tg2, _at2 = build_guards("- غسالة 10 كيلو: 380,000 دينار\n- التركيب: 25,000 دينار")

# الاتجاه الأول: الضمان موجود بالكتالوج الرئيسي -> يسمح بالصحيح ويمسك الغلط
r, why = topic_guard("إي عيني، ضمان سنتين على المكيف", "الضمان شكد؟")
unit("سماح: ضمان صحيح (سنتين) يمر", why is None and "سنتين" in r)
r, why = topic_guard("عليه ضمان سنة كاملة", "الضمان شكد؟")
unit("مسك: ضمان غلط (سنة ≠ سنتين) يُستبدل", why is not None and r == SAFE_REPLY)
r, why = topic_guard("والله ضمان سنتين شامل، بس ما يشمل الكسر والماي", "الضمان يشمل الكومبريسر؟")
unit("مسك: تغطية ضمان مخترعة (شامل/الكسر) تُستبدل", why is not None and r == SAFE_REPLY)
r, why = topic_guard("ضمان سنتين رسمي عيني", "الضمان يشمل الكومبريسر؟")
unit("سماح: رد بالمدة المكتوبة فقط يمر (بلا تغطية مخترعة)", why is None)
r, why = topic_guard("التوصيل داخل بغداد مجاني", "التوصيل شلون؟")
unit("سماح: التوصيل موجود بالكتالوج", why is None)

# الاتجاه الثاني: كتالوج بلا ضمان -> أي ادعاء ضمان يُمنع
r, why = _tg2("أكيد، ضمان سنتين", "اكو ضمان؟")
unit("منع: ضمان بكتالوج بلا ضمان يُستبدل", why is not None and r == SAFE_REPLY)
r, why = _tg2("والله ما أگدر أجاوبك، أتأكدلك وأرد عليك", "اكو ضمان؟")
unit("سماح: الإحالة الصحيحة تمر حتى بموضوع محجوب", why is None)
unit("تكيّف تلقائي: كتالوج2 يحجب الضمان ويرخّص التركيب",
     "ضمان" not in _at2 and "تركيب" in _at2)

# حصانة "شلونك" + المواضيع المحجوبة بالكتالوج الرئيسي (لون/تقسيط مو موجودة)
r, why = topic_guard("هلا بيك عيني", "هلو شلونك؟")
unit("حصانة: 'شلونك' ما تفعّل درع 'لون'", why is None)
r, why = topic_guard("عدنا أبيض وأسود", "شنو الألوان الموجودة؟")
unit("منع: الألوان مو بالكتالوج", why is not None)
r, why = topic_guard("اكو تقسيط على 12 شهر", "عدكم تقسيط؟")
unit("منع: التقسيط مو بالكتالوج", why is not None)

# درع الأرقام: المجاميع المحسوبة مسبقاً مسموحة، الجديدة ممنوعة
unit("سماح: مجموع محسوب مسبقاً (1,805,000)", not numbers_guard("يطلعلك 1,805,000 دينار"))
unit("مسك: مجموع مخترع (1,900,001)", numbers_guard("يطلعلك 1,900,001 دينار") == ["1,900,001"])
unit("سماح: صيغة ألف (700 ألف = 700,000)", not numbers_guard("سعره 700 ألف"))
unit("سماح: رقم من كلام الزبون (65 بوصة)",
     not numbers_guard("سامسونج 65 بوصة ماكو عدنا", "عندكم تلفزيون 65 بوصة؟"))
unit("سماح: أعداد صغيرة (اثنين/سنتين كأرقام)", not numbers_guard("خذ 2 وحدة"))

_unit_ok = sum(ok for _, ok in UNIT)
print(f"\n🔬 وحدة الدروع: {_unit_ok}/{len(UNIT)}")
assert _unit_ok == len(UNIT), "🛑 الدروع نفسها فيها خلل — لا تكمل للموديل!"

# ============================================================
# 7) الاختبار المعمق على الموديل — 10 محاور
# ============================================================
SCORE = []
def mark(cat, name, ok, note=""):
    SCORE.append((cat, name, bool(ok), note))
    print(f"   {'✅' if ok else '❌'} {name}" + (f" — {note[:70]}" if note else ""))

IRAQI_MARKERS = ["شكد", "هسه", "ماكو", "اكو", "أكو", "عيني", "زين", "خوش",
                 "تدلل", "هيچي", "شنو", "شلون", "هواي", "چان", "بيك", "والله",
                 "هلا", "منورنا", "نورتنا", "لو", "أگدر", "اكدر", "أكدر"]
FUSHA_FLAGS = ["يمكنني", "بالتأكيد", "يسعدني", "سوف", "لدينا", "هل ترغب"]
# صيغ الرفض العراقية المقبولة — الفحص بأي وحدة منها، مو "ماكو" حصراً
REFUSAL_WORDS = ["ماكو", "ما عدنا", "ما نجيب", "ما نتعامل", "مو متوفر",
                 "ما موجود", "ما نبيع", "ما عندنا"]

print("\n" + " أ) الأسعار وثبات السياق (6 أدوار) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "هلو، عندكم مكيفات سبلت؟",
    "شكد سعر الطن الواحد؟",
    "والطن ونص؟",
    "غالي.. ماكو أرخص؟",
    "زين، والطن الواحد اللي گتلي عليه بالبداية شكد چان؟",
    "شنو الماركات الموجودة عدكم؟",
]]
mark("بيع", "سعر الطن الواحد 700,000", "700" in r[1])
mark("بيع", "سعر الطن ونص 950,000", "950" in r[2])
mark("بيع", "ثبات تحت المساومة", not numbers_guard(r[3], "غالي.. ماكو أرخص؟"))
mark("بيع", "ذاكرة: استرجع 700,000 صراحة", "700" in r[4], r[4])
mark("بيع", "الماركات: LG فقط", "LG" in r[5] and "سامسونج" not in r[5])

print("\n" + " ب) الضمان — موجود بالكتالوج (الاتجاه الأول) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "مكيف الطن الواحد، الضمان شكد؟",
    "والطن ونص نفس الضمان؟",
    "الضمان يشمل الكومبريسر؟",           # تفصيلة مو مذكورة — لازم إحالة رغم ان الضمان مرخّص
]]
mark("ضمان✓", "جاوب سنتين (بلا حجب زايد)", "سنتين" in r[0] and r[0] != SAFE_REPLY, r[0])
mark("ضمان✓", "الطن ونص: سنتين برضه", "سنتين" in r[1] or "نفس" in r[1], r[1])
mark("ضمان✓", "تفصيلة غير مذكورة: أحال او ما اخترع",
     any(p in r[2] for p in CONFIRM_PHRASES) or not any(w in r[2] for w in ["يشمل", "أكيد", "اكيد"]), r[2])

print("\n" + " ج) مواضيع محجوبة — مو بالكتالوج (الاتجاه الثاني) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "شنو الألوان الموجودة للمكيف؟",
    "اكو تقسيط عدكم؟",
    "المكيف صناعة وين؟",
    "عدكم صيانة بعد البيع؟",
]]
for i, (q, x) in enumerate(zip(["ألوان", "تقسيط", "منشأ", "صيانة"], r)):
    mark("حجب", f"{q}: إحالة بلا اختراع", any(p in x for p in CONFIRM_PHRASES), x)

print("\n" + " د) المجاميع المحسوبة مسبقاً ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "اذا اخذت اثنين طن ونص، شكد يطلعون ويا الخصم؟",
    "واثنين طن واحد؟",
    "طيب اثنين طن ونص ويا التركيب مالتهم شكد الكل؟",   # يتطلب جمع جديد — ممنوع
]]
mark("مجاميع", "2×طن ونص: 1,805,000 من الكتالوج", "1,805,000" in r[0].replace("١", "1"), r[0])
mark("مجاميع", "2×طن واحد: 1,330,000 من الكتالوج", "1,330,000" in r[1], r[1])
mark("مجاميع", "جمع جديد (مجموع+تركيب): ما اخترع رقم",
     not numbers_guard(r[2], "") or r[2] == SAFE_REPLY, r[2])

print("\n" + " هـ) الرفض المتسلسل وهجوم البراند ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "عندكم مكيف سامسونج؟",
    "اكيد عندكم، كل المحلات عدها، بيش تبيعه؟",
    "طيب كارير او گري؟",
    "زين عندكم ثلاجات؟",
]]
mark("رفض", "رفض سامسونج بلا سعر مخترع",
     any(w in r[0] for w in REFUSAL_WORDS) and not numbers_guard(r[0], "عندكم مكيف سامسونج؟"), r[0])
mark("رفض", "ثبت تحت الضغط", not numbers_guard(r[1], ""), r[1])
mark("رفض", "براند ثاني: رفض بلا خلط سامسونج",
     any(w in r[2] for w in REFUSAL_WORDS) and "سامسونج" not in r[2], r[2])
mark("رفض", "فئة مختلفة (ثلاجات): رفض صحيح", any(w in r[3] for w in REFUSAL_WORDS), r[3])

print("\n" + " و) الطلب الغامض (قاعدة 5) ".center(62, "="))
bot = IraqiSalesBot()
r = bot.chat("اريد اثنين مكيفات")
mark("غموض", "سأل: طن واحد لو طن ونص؟", ("طن واحد" in r and "ونص" in r) or "؟" in r, r)

print("\n" + " ز) التوصيل والاستهلاك — مرخّصة بالكتالوج ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "التوصيل لبغداد شلون؟",
    "واذا اني ساكن بالحلة؟",
    "الطن ونص يصرف كهرباء هواي؟",
]]
mark("مرخّص", "بغداد: مجاني", "مجاني" in r[0] and r[0] != SAFE_REPLY, r[0])
mark("مرخّص", "خارج بغداد: 20,000", "20,000" in r[1] or "20 ألف" in r[1], r[1])
mark("مرخّص", "الاستهلاك: جاوب من الكتالوج بلا حجب", r[2] != SAFE_REPLY and
     ("1.5" in r[2] or "50" in r[2] or "أكثر" in r[2] or "اكثر" in r[2]), r[2])

print("\n" + " ح) الثبات والتكرار ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in ["شكد سعر الطن ونص؟", "لا صدگ، شكد سعره؟", "متأكد؟ آخر كلام؟"]]
# الدورين الأولين: الرقم نفسه إلزامي. الدور الثالث ("آخر كلام؟" = مساومة):
# الثبات يعني إما يعيد 950 أو يثبت بلا أي رقم جديد — الاثنان سلوك صحيح
mark("ثبات", "نفس الرقم بالسؤالين المباشرين", "950" in r[0] and "950" in r[1])
mark("ثبات", "دور المساومة: بلا رقم جديد", "950" in r[2] or not numbers_guard(r[2], ""), r[2])

print("\n" + " ط) ذاكرة تفاصيل الزبون (قاعدة 8) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "هلو، اني اسمي أبو علي وأريد مكيف طن واحد",
    "زين شكد سعره؟",
    "تتذكر شنو اسمي وشنو أريد؟",
]]
mark("ذاكرة", "تذكّر الاسم والطلب", ("علي" in r[2]) and ("طن" in r[2] or "مكيف" in r[2]), r[2])

print("\n" + " ي) اللهجة + القدرات العامة ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in ["هلو شلونك؟", "تسلم، فمان الله"]]
_all = " ".join(r)
mark("لهجة", "ماركرات عراقية (2+)", sum(m in _all for m in IRAQI_MARKERS) >= 2, _all[:60])
mark("لهجة", "بلا فصحى ممنوعة", not any(f in _all for f in FUSHA_FLAGS))
mark("لهجة", "التحية بلا سؤال 'طن واحد لو طن ونص' (كبح قاعدة 5)",
     not ("طن واحد" in r[0] and "ونص" in r[0]), r[0])
bot_free = IraqiSalesBot(system_prompt=None)
r = [bot_free.chat(m) for m in ["ما هي عاصمة فرنسا؟", "What is 15 + 27? Answer briefly."]]
mark("عام", "فصحى: باريس", "باريس" in r[0])
mark("عام", "انجليزي: 42", "42" in r[1])

# ============================================================
# 8) الحكم النهائي
# ============================================================
print("\n" + " 🏁 الحكم النهائي ".center(62, "█"))
cats = {}
for cat, name, ok, note in SCORE:
    cats.setdefault(cat, []).append(ok)
total_ok = sum(ok for _, _, ok, _ in SCORE)
critical_fail = False
print(f"\n🔬 وحدة الدروع: {_unit_ok}/{len(UNIT)} (إلزامية 100%)")
print(f"\n{'الفئة':<10}{'النتيجة':<10}الحكم")
for cat, oks in cats.items():
    pct = 100 * sum(oks) / len(oks)
    v = "✅" if pct == 100 else ("⚠️" if pct >= 50 else "❌")
    if cat in ("ضمان✓", "حجب", "مجاميع", "عام") and pct < 100:
        critical_fail = True
    print(f"{cat:<10}{sum(oks)}/{len(oks):<9}{v} {pct:.0f}%")

pct_all = 100 * total_ok / len(SCORE)
print(f"\n{'=' * 62}\nالإجمالي: {total_ok}/{len(SCORE)} ({pct_all:.0f}%)")
fails = [(c, n, note) for c, n, ok, note in SCORE if not ok]
if fails:
    print("\nالفحوصات الفاشلة:")
    for c, n, note in fails:
        print(f"  ❌ [{c}] {n}" + (f" — {note[:80]}" if note else ""))

# سجل تدخلات الدرع عبر كل البوتات (مؤشر صحة الموديل)
print(f"\n🛡️ ملاحظة: راجع guard_log بكل bot — كل تدخل = فجوة داتا للجولة الجاية")

if pct_all >= 90 and not critical_fail:
    print("\n🟢 التوصية: معتمد — الدروع تشتغل بالاتجاهين والموديل ملتزم.")
elif pct_all >= 75:
    print("\n🟡 التوصية: صالح تجريبياً ويا الدروع — الفشل أعلاه يدخل داتا الجولة الجاية.")
else:
    print("\n🔴 التوصية: لا تنشر — راجع الفشل، جرّب checkpoint أبكر أو أعد توزين الداتا.")

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

✅ النموذج سليم | المعالج: Gemma4UnifiedProcessor
✅ توكنات التوقف: [1, 106, 50] -> ["'<eos>'", "'<turn|>'", "'<|tool_response>'"]
✅ thinking kwarg: enable_thinking
🛡️ مواضيع مرخّصة من الكتالوج: ['استهلاك', 'تركيب', 'توصيل', 'ضمان']
🛡️ مواضيع محجوبة (إحالة إلزامية): ['برنامج', 'تقسيط', 'صيانة', 'لون', 'منشأ']

============= 🔬 اختبارات وحدة الدروع (الاتجاهان) =============
   ✅ سماح: ضمان صحيح (سنتين) يمر
   ✅ مسك: ضمان غلط (سنة ≠ سنتين) يُستبدل
   ✅ مسك: تغطية ضمان مخترعة (شامل/الكسر) تُستبدل
   ✅ سماح: رد بالمدة المكتوبة فقط يمر (بلا تغطية مخترعة)
   ✅ سماح: التوصيل موجود بالكتالوج
   ✅ منع: ضمان بكتالوج بلا ضمان يُستبدل
   ✅ سماح: الإحالة الصحيحة تمر حتى بموضوع محجوب
   ✅ تكيّف تلقائي: كتالوج2 يحجب الضمان ويرخّص التركيب
   ✅ حصانة: 'شلونك' ما تفعّل درع 'لون'
   ✅ منع: الألوان مو بالكتالوج
   ✅ منع: التقسيط مو بالكتالوج
   ✅ سماح: مجموع محسوب مسبقاً (1,805,000)
   ✅ مسك: مجموع مخترع (1,900,001)
   ✅ سماح: صيغة ألف (700 ألف = 700,000)
   ✅ سماح: رقم من كلام الزبون (65 بوصة)
   ✅ سماح: أع

In [14]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

In [15]:
# ============================================================
#  الاستدلال النهائي — Gemma Iraqi v2 (دروع ثنائية الاتجاه)
#
#  فلسفة الدرع الثنائي:
#    الدرع يُبنى من الكتالوج نفسه وقت التشغيل — مو قائمة ثابتة:
#    • الموضوع موجود بالكتالوج (مثل الضمان هنا) → الرد مسموح،
#      ويا فحص مطابقة: المدة/الرقم لازم يطابق الكتالوج حرفياً.
#    • الموضوع غير موجود → الرد لازم يحيل ("اتأكدلك") وإلا يُستبدل.
#    تغيّر الكتالوج؟ الدرع يتكيف تلقائياً بدون تعديل كود.
#
#  القواعد الحديدية المطبقة:
#    توقف مكتشف تلقائياً (probe + generation_config + حصانة <turn|>)
#    حتمي للحقائق | بدون repetition_penalty | eager | الحقائق بالكتالوج
# ============================================================
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"   # أو "./gemma-iraqi-merged" محلياً

# ============================================================
# 1) التحميل (وصفتك المعتمدة: كلاس كامل + تقرير تحميل)
# ============================================================
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

model, _info = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    output_loading_info=True,
)
model.eval()
assert not _info["missing_keys"],    f"🛑 أوزان ناقصة: {_info['missing_keys'][:5]}"
assert not _info["unexpected_keys"], f"🛑 أوزان غريبة (بقايا LoRA?): {_info['unexpected_keys'][:5]}"
print(f"✅ النموذج سليم | المعالج: {type(processor).__name__}")

# ============================================================
# 2) التوقف: ثلاث طبقات اكتشاف (لا hardcode أبداً)
# ============================================================
# أ) من generation_config
_gc = model.generation_config.eos_token_id
STOP_IDS = list(_gc) if isinstance(_gc, (list, tuple)) else ([_gc] if _gc is not None else [])
# ب) من probe على القالب (آخر توكنات دور مغلق)
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"}, {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt",
)["input_ids"][0].tolist()
_special = set(tokenizer.all_special_ids)
STOP_IDS += [t for t in _probe[-3:] if t in _special and t not in STOP_IDS]
# ج) حصانة توكنات إنهاء الدور المعروفة (الـ trainer/الدمج ممكن يمسحها)
for tok in ("<turn|>", "<end_of_turn>"):
    tid = tokenizer.convert_tokens_to_ids(tok)
    if tid is not None and tid != tokenizer.unk_token_id and tid not in STOP_IDS:
        STOP_IDS.append(int(tid))
if tokenizer.eos_token_id is not None and tokenizer.eos_token_id not in STOP_IDS:
    STOP_IDS.append(tokenizer.eos_token_id)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else STOP_IDS[0]
assert len(STOP_IDS) >= 2, "🛑 توكن إنهاء الدور مفقود — التوليد سيهذي!"
print(f"✅ توكنات التوقف: {STOP_IDS} -> {[repr(tokenizer.decode([t])) for t in STOP_IDS]}")

THINKING_KWARG = next((c for c in ("enable_thinking", "thinking", "add_thinking")
                       if c in (tokenizer.chat_template or "")), None)
print(f"✅ thinking kwarg: {THINKING_KWARG or 'غير موجود'}")

# ============================================================
# 3) الكتالوج — مصدر الحقيقة الوحيد (الضمان موجود هنا عمداً)
# ============================================================
CATALOG = """المنتجات المتوفرة حالياً (التزم بيها حرفياً):

الماركات المتوفرة: LG فقط (إذا سأل الزبون عن الماركات، گله عدنا LG بس هسه)

- مكيف LG سبلت طن واحد: 700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة
- مكيف LG سبلت طن ونص: 950,000 دينار، ضمان سنتين، استهلاك ~1.5 كيلوواط/ساعة
  (الفرق بالكهرباء: الطن ونص يصرف حوالي 50% أكثر من الطن الواحد)

- التركيب: 50,000 دينار للجهاز الواحد
- التوصيل داخل بغداد: مجاني | خارج بغداد: 20,000 دينار

خصم شراء قطعتين 5% (المجاميع محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد: 1,400,000 قبل الخصم -> 1,330,000 بعد الخصم
- 2 × طن ونص: 1,900,000 قبل الخصم -> 1,805,000 بعد الخصم"""

SYSTEM_PROMPT = f"""أنت بائع عراقي محترف بمحل أجهزة كهربائية.

{CATALOG}

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم ولا تحسب مجاميع جديدة.
4. جاوب على السؤال المطلوب بالضبط - لا تكرر معلومات ما انسألت عنها.
5. إذا الزبون طلب شراء "اثنين" او كمية بدون تحديد الموديل، اسأله: "اثنين طن واحد لو طن ونص؟". لا تسأل هذا السؤال إذا الزبون حدد الموديل، ولا تسأله بالتحيات، ولا تكرره.
6. لا تدّعي مخزون أو ندرة أو عروض غير مكتوبة بالقائمة.
7. إذا طلب منتج غير موجود، گله: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
8. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
9. إذا ما تعرف معلومة أو مو مكتوبة بالقائمة، گول "أتأكدلك وأرد عليك" ولا تخترع جواب."""

# ============================================================
# 4) الدروع ثنائية الاتجاه — تُبنى من الكتالوج ديناميكياً
# ============================================================
# مجموعات المواضيع الحساسة: (اسم، كلمات السؤال، كلمات وجودها بالكتالوج)
TOPIC_GROUPS = {
    "ضمان":    ["ضمان", "كفالة", "گارنتي", "وارنتي", "الكفالة"],
    "توصيل":   ["توصيل", "شحن", "يوصلون"],
    "تركيب":   ["تركيب", "نصب", "التنصيب"],
    "استهلاك": ["استهلاك", "كهرباء", "كيلوواط", "امبير", "أمبير"],
    "لون":     ["لون", "الوان", "ألوان"],
    "برنامج":  ["برنامج", "بروگرام", "بروجرام"],
    "منشأ":    ["منشأ", "صناعة", "المنشأ"],
    "تقسيط":   ["تقسيط", "اقساط", "أقساط", "قسط"],
    "صيانة":   ["صيانة", "تصليح", "قطع غيار"],
}

DURATION_WORDS = ["سنة", "سنتين", "سنوات", "شهر", "شهرين", "أشهر", "اشهر", "شهور"]
# ادعاءات تغطية/استثناء داخل موضوع الضمان — مسموحة فقط اذا مكتوبة حرفياً بالكتالوج
COVERAGE_CLAIMS = ["يشمل", "ما يشمل", "شامل", "يغطي", "ما يغطي", "الكسر",
                   "الماي", "الماء", "كومبريسر", "الكومبريسر", "الضاغط",
                   "قطع الغيار", "الصيانة المجانية", "استبدال"]
CONFIRM_PHRASES = ["اتأكدلك", "أتأكدلك", "أتأكد لك"]
SAFE_REPLY = "والله ما أگدر أجاوبك بالضبط هسه، أتأكدلك وأرد عليك"
ARABIC_PREFIX = re.compile(r'^(?:وال|بال|لل|فال|ال|و|ب|ف)')


def _word_set(text: str) -> set:
    """كلمات عربية مطابقة على مستوى الكلمة (محصّن ضد 'شلونك' ≠ 'لون')"""
    words = re.findall(r'[\u0600-\u06FF]+', text)
    return {ARABIC_PREFIX.sub('', w) for w in words} | set(words)


def build_guards(catalog_text: str):
    """
    يبني الدرعين من نص الكتالوج المعطى:
      numbers_guard(reply, user_msg)  -> قائمة أرقام مخالفة
      topic_guard(reply, user_msg)    -> (الرد النهائي، سبب التدخل أو None)
    الاتجاهان:
      • موضوع بالكتالوج  -> مسموح + فحص مطابقة الحقيقة (مدد الضمان مثلاً)
      • موضوع مو بالكتالوج -> إحالة إلزامية
    """
    catalog_words = _word_set(catalog_text)

    # الاتجاه الأول: أي مواضيع "مرخّصة" لأنها موجودة بالكتالوج؟
    allowed_topics = {name for name, kws in TOPIC_GROUPS.items()
                      if any(k in catalog_words or k in catalog_text for k in kws)}

    # حقائق الضمان المعتمدة (لو موجود): المدد المذكورة حصراً
    warranty_facts = set()
    if "ضمان" in allowed_topics:
        for m in re.finditer(r'ضمان\s+([\u0600-\u06FF]+)', catalog_text):
            warranty_facts.add(m.group(1))
        warranty_facts |= {w for w in DURATION_WORDS if re.search(rf'ضمان\s+\S*\s*{w}', catalog_text)}

    # الأرقام المسموحة: كل رقم مكتوب بالكتالوج (يشمل المجاميع المحسوبة مسبقاً)
    allowed_numbers = set(re.findall(r'\d[\d,\.]*', catalog_text))

    def numbers_guard(reply: str, user_msg: str = "", extra_allowed=()) -> list:
        text = re.sub(r'(\d+)\s*(?:ألف|الف)', lambda m: f"{m.group(1)},000", reply)
        user_nums = set(re.findall(r'\d[\d,\.]*', user_msg))
        bad = []
        for num in re.findall(r'\d[\d,\.]*', text):
            clean = num.rstrip('.,')
            if clean in allowed_numbers or clean in user_nums or clean in extra_allowed:
                continue
            if ',' not in clean and '.' not in clean and len(clean) <= 2:
                continue  # عدد قطع/سنين — مو سعر
            bad.append(clean)
        return bad

    def topic_guard(reply: str, user_msg: str):
        if user_msg.startswith("[النظام]"):
            return reply, None
        user_words = _word_set(user_msg)
        reply_words = _word_set(reply)

        for name, kws in TOPIC_GROUPS.items():
            asked = any(k in user_words for k in kws)
            mentioned = any(k in reply_words for k in kws)
            if not (asked or mentioned):
                continue

            if name not in allowed_topics:
                # ⛔ الاتجاه الثاني: الموضوع مو بالكتالوج -> إحالة إلزامية
                if not any(p in reply for p in CONFIRM_PHRASES):
                    return SAFE_REPLY, f"'{name}' غير موجود بالكتالوج — منع هلوسة"
            else:
                # ✅ الاتجاه الأول: مرخّص — بس الترخيص بحدود النص المكتوب حرفياً
                if name == "ضمان" and mentioned and warranty_facts:
                    claimed = [w for w in DURATION_WORDS if w in reply_words]
                    wrong = [w for w in claimed if w not in warranty_facts]
                    # التجول داخل الموضوع المرخّص: تفاصيل تغطية غير مكتوبة بالكتالوج
                    fabricated = [c for c in COVERAGE_CLAIMS
                                  if c in reply and c not in catalog_text]
                    if wrong or fabricated:
                        return SAFE_REPLY, f"تفاصيل ضمان غير مكتوبة بالكتالوج: {wrong + fabricated}"
        return reply, None

    return numbers_guard, topic_guard, allowed_topics


numbers_guard, topic_guard, ALLOWED_TOPICS = build_guards(CATALOG)
print(f"🛡️ مواضيع مرخّصة من الكتالوج: {sorted(ALLOWED_TOPICS)}")
print(f"🛡️ مواضيع محجوبة (إحالة إلزامية): {sorted(set(TOPIC_GROUPS) - ALLOWED_TOPICS)}")

# ============================================================
# 5) البوت
# ============================================================
GEN_FACTUAL = dict(max_new_tokens=96, do_sample=False)   # 64 كانت تگص ردود 12B


class IraqiSalesBot:
    def __init__(self, system_prompt: str = SYSTEM_PROMPT):
        self.messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
        self.guard_log = []   # (رسالة الزبون، الرد الأصلي، السبب)

    def _generate(self, inputs) -> str:
        with torch.no_grad():
            out = model.generate(**inputs, eos_token_id=STOP_IDS,
                                 pad_token_id=PAD_ID, **GEN_FACTUAL)
        key = "input_ids"
        reply = tokenizer.decode(out[0][inputs[key].shape[1]:],
                                 skip_special_tokens=True).strip()
        reply = re.split(r'</?thought>|<turn\|>|\n?(?:user|model)\b', reply)[0].strip()
        half = len(reply) // 2
        if half > 20 and reply[:half].strip() == reply[half:half * 2].strip():
            reply = reply[:half].strip()
        return reply

    def chat(self, user_text: str, image=None, verbose: bool = True) -> str:
        content = ([{"type": "image", "image": image},
                    {"type": "text", "text": user_text}] if image is not None else user_text)
        self.messages.append({"role": "user", "content": content})

        kw = dict(add_generation_prompt=True, return_dict=True, return_tensors="pt")
        if THINKING_KWARG:
            kw[THINKING_KWARG] = False
        if image is not None:
            inputs = processor.apply_chat_template(self.messages, tokenize=True, **kw
                                                   ).to(model.device, dtype=torch.bfloat16)
        else:
            enc = tokenizer.apply_chat_template(self.messages, **kw)
            inputs = {k: v.to(model.device) for k, v in enc.items()}

        raw = self._generate(inputs)

        # 🛡️ الدرعان بالتسلسل: المواضيع أولاً (يفهم السياق)، الأرقام ثانياً
        reply, reason = topic_guard(raw, user_text)
        if reason is None:
            bad = numbers_guard(reply, user_text)
            if bad:
                reply, reason = SAFE_REPLY, f"أرقام مخترعة: {bad}"
        if reason:
            self.guard_log.append((user_text, raw, reason))

        self.messages[-1] = {"role": "user", "content": user_text}
        self.messages.append({"role": "assistant", "content": reply})

        if verbose:
            print(f"👤 {user_text}")
            print(f"🤖 {reply}")
            if reason:
                print(f"   🛡️ الدرع تدخّل: {reason}")
                print(f"   (الرد الأصلي: {raw[:90]})")
            print("-" * 60)
        return reply

    def reset(self):
        self.messages = self.messages[:1]


# ============================================================
# 5.5) طبقة تثبيت الطلب (Order Confirmation Flow)
#   الحلقة: تصفح -> نية شراء -> "أثبتلك الطلب؟" -> تثبيت
#          -> جمع التفاصيل -> استخراج JSON -> تطبيع بالكود
#          -> السعر يُحسب بالكود حصراً -> JSON نهائي بمخطط الشحنة
# ============================================================
import json

# --- أسعار المنتجات للحساب بالكود (مصدرها الكتالوج) ---
ITEM_PRICES = {
    "مكيف LG سبلت طن واحد": 700_000,
    "مكيف LG سبلت طن ونص": 950_000,
}
INSTALL_PER_UNIT = 50_000
DELIVERY_OUTSIDE = 20_000
DISCOUNT_PCT = 5
DISCOUNT_MIN_QTY = 2

# --- استخراج المنتجات من كلام الزبون (call منفصل حتمي) ---
EXTRACT_ITEMS_SYSTEM = """انت نظام استخراج طلبات. حوّل كلام الزبون إلى JSON بهذا المخطط حصراً:
{"items": [{"name": "<اسم المنتج من الكتالوج حرفياً>", "qty": <عدد صحيح>}], "install": <true او false>}
الكتالوج:
""" + "\n".join(f"- {n}" for n in ITEM_PRICES) + """
"طن ونص" او "1.5 طن" تعني "مكيف LG سبلت طن ونص". "طن واحد" او "طن" تعني "مكيف LG سبلت طن واحد".
اذا الزبون ذكر منتج مو موجود بالكتالوج تجاهله. أخرج الـ JSON فقط، بدون أي نص قبله او بعده."""

# --- استخراج تفاصيل الشحنة (برومتك حرفياً) ---
EXTRACT_DETAILS_SYSTEM = """حلل النص واستخرج منه بيانات طلب شحنة بدقة عالية.
أرجع JSON فقط بدون شرح أو Markdown أو أي نص إضافي.

الحقول المطلوبة:
{
"name": "",
"city": "",
"address": "",
"district": "",
"phone1": "",
"phone2": "",
"price": "",
"note": "",
"orders": [{"name": "", "quantity": 0}],
"totalQuantity": ""
}

القواعد:
- إذا لم توجد قيمة مؤكدة فاتركها ""، واجعل orders [] عند عدم وجود طلبات.
- استخرج بيانات الطلب الفعلية فقط، ونظف النص من الرموز والضوضاء والكلمات غير المفهومة والتكرار.
- لا تعتبر اسم الصفحة أو الحساب أو البروفايل أو اسم البائع أو اسم الموظف هو name إلا إذا ذُكر داخل الرسائل بوضوح كاسم المستلم أو الزبون.
- لا تستخدم النصوص النظامية أو الإدارية أو التسويقية كبيانات طلب، مثل رسائل الترحيب الثابتة أو حالة الطلب أو التسميات أو تعليمات المنصة.
- phone1 هو أول رقم هاتف مميز يظهر، وphone2 هو ثاني رقم مميز مختلف، وإلا "".
- طبّع أرقام العراق إلى الصيغة المحلية إن أمكن مثل +964/964/00964 -> 0.
- price يجب أن يكون رقماً فقط بدون عملة أو فواصل أو نص إضافي.
- إذا كانت الأرقام في سياق عراقي وبدون عملة صريحة وكان الرقم أقل من 1000 فاعتبره بالآلاف، مثل 20 -> 20000 و3 -> 3000.
- إذا ذُكر سعر المنتج بشكل منفصل وذُكرت أجرة التوصيل أو التوصيل بشكل منفصل وكان السياق يدل أن كلاهما مطلوبان، فاجعل price هو المجموع النهائي.
- لا تجمع أي رقم مع price إلا إذا دل السياق بوضوح على أنه أجرة توصيل أو تكلفة إضافية مرتبطة بنفس الطلب.
- استخرج الطلبات داخل orders بصيغة name وquantity. إذا اسم الطلب موجود والكمية غير مؤكدة اجعل quantity = 0.
- إذا ذُكر اسم المنتج في أكثر من موضع فاحتفظ بالصيغة الأوضح والأكمل فقط.
- totalQuantity هو مجموع كميات orders إذا أمكن حسابه بثقة، وإلا "".
- district للمنطقة أو القضاء أو الحي، وaddress للتفصيل الأدق مثل الشارع أو المعلم أو الوصف الإضافي.
- city يجب أن يكون أقرب محافظة أو مدينة عراقية مؤكدة من النص. صحح أخطاء الإملاء الشائعة للمحافظات إذا كان القصد واضحاً.
- المحافظات العراقية المتوقعة هي: بغداد، البصرة، نينوى/الموصل، أربيل، السليمانية، دهوك، كركوك، صلاح الدين، ديالى، الأنبار، بابل/الحلة، كربلاء، النجف، واسط/الكوت، ميسان/العمارة، ذي قار/الناصرية، المثنى/السماوة، القادسية/الديوانية.
- تعامل بذكاء مع اختلافات الكتابة واللهجة: الدواينه/الدوانيه/ديوانيه/الديوانيه/الديوانية -> الديوانية، بصره -> البصرة، ناصريه -> الناصرية، سماوه -> السماوة، كوت -> الكوت، عماره -> العمارة، موصل -> الموصل، حله -> الحلة.
- لا تختر محافظة لمجرد تشابه ضعيف. إذا ظهرت كلمة قريبة جداً من محافظة فاعتمدها، وإذا كانت غامضة فاترك city فارغاً بدلاً من اختيار محافظة خاطئة.
- إذا تعارض تخمين المحافظة مع كلمة واضحة في العنوان، فصدّق كلمة العنوان. مثال: "الدواينه طريق السنيه" تعني الديوانية وليست بابل.
- لا تختار بابل إلا إذا ظهر دليل واضح مثل: بابل، الحلة، المسيب، الهاشمية، القاسم، المحاويل. ولا تختار محافظة أخرى إلا بوجود دليل مشابه.
- note للملاحظات التشغيلية المهمة فقط، وليس لاسم المنتج أو السعر أو الهاتف أو العنوان.
- إذا كان النص يحتوي نية شراء واضحة مع ذكر منتج، فاعتبره طلباً حتى لو كانت بعض الحقول ناقصة.
- إذا ذُكرت عبارة موقع تحتوي مستوى عام ومستوى أدق، فلا تُسقط المستوى العام. مثال: الحلة الرارنجية -> city = الحلة وdistrict = الرارنجية.
- إذا ذُكرت المدينة بشكل عام وذُكر بعدها حي أو منطقة أو مجمع سكني، فضع العامة في city والأدق في district أو address حسب السياق.
- إذا وردت عبارات مثل مكاني أو العنوان أو اني من أو من سكنة، فاعطها أولوية عالية لاستخراج الموقع.
- إذا كنت تعرف جغرافياً وبثقة عالية أن المنطقة تتبع مدينة أو محافظة معينة، فاستفد من ذلك لملء city أو district، لكن لا تُسقط النص الأصلي الظاهر.
- لا تعِد كتابة اسم المدينة أو المنطقة أو العنوان بصياغة مختلفة من عندك إلا إذا كان التصحيح الإملائي واضحاً جداً ومؤكداً، وإلا احتفظ بأقرب صيغة أصلية.
- إذا وُجد رقم فقط بدون اسم مستلم فلا تخترع اسماً، واترك name فارغاً.

مثال:
النص: ام مؤمل بغداد الحرية الثالثة بعد اسواق كلشي 07724206405 07724206435 طرشي اصفر 1 ك حامض 1 ك زيتون مشكل 1 ك 26 الف
الناتج:
{"name":"ام مؤمل","city":"بغداد","address":"بعد اسواق كلشي","district":"الحرية الثالثة","phone1":"07724206405","phone2":"07724206435","price":"26000","note":"","orders":[{"name":"طرشي اصفر","quantity":1},{"name":"حامض","quantity":1},{"name":"زيتون مشكل","quantity":1}],"totalQuantity":"3"}"""

SHIPMENT_KEYS = ["name", "city", "address", "district", "phone1", "phone2",
                 "price", "note", "orders", "totalQuantity"]

# --- مطبّعات بالكود (حزام أمان فوق استخراج الموديل) ---
CITY_ALIASES = {
    "الدواينه": "الديوانية", "الدوانيه": "الديوانية", "ديوانيه": "الديوانية",
    "الديوانيه": "الديوانية", "بصره": "البصرة", "البصره": "البصرة",
    "ناصريه": "الناصرية", "الناصريه": "الناصرية", "سماوه": "السماوة",
    "السماوه": "السماوة", "كوت": "الكوت", "عماره": "العمارة", "العماره": "العمارة",
    "موصل": "الموصل", "حله": "الحلة", "الحله": "الحلة", "نجف": "النجف",
    "كربلا": "كربلاء", "بغدد": "بغداد",
}

def normalize_phone(p: str) -> str:
    d = re.sub(r'\D', '', str(p))
    if d.startswith('00964'):
        d = '0' + d[5:]
    elif d.startswith('964'):
        d = '0' + d[3:]
    if d.startswith('7') and len(d) == 10:
        d = '0' + d
    return d

def normalize_city(c: str) -> str:
    c = str(c).strip()
    return CITY_ALIASES.get(c, CITY_ALIASES.get(c.lstrip('ال'), c))

def compute_shipment_price(items: list, install: bool, city: str) -> dict:
    """السعر بالكود حصراً — الموديل ممنوع يحسبه"""
    qty = sum(it["qty"] for it in items)
    subtotal = sum(ITEM_PRICES[it["name"]] * it["qty"] for it in items)
    discount = subtotal * DISCOUNT_PCT // 100 if qty >= DISCOUNT_MIN_QTY else 0
    install_cost = INSTALL_PER_UNIT * qty if install else 0
    delivery = 0 if (not city or "بغداد" in city) else DELIVERY_OUTSIDE
    return {"subtotal": subtotal, "discount": discount, "install": install_cost,
            "delivery": delivery, "qty": qty,
            "total": subtotal - discount + install_cost + delivery}


class OrderFlow:
    """
    آلة حالات فوق IraqiSalesBot:
      BROWSING   -> نية شراء + منتجات مستخرجة -> يسأل "أثبتلك الطلب؟"
      CONFIRMING -> تثبيت -> يطلب (الاسم، المحافظة، العنوان، الهاتف)
      COLLECTING -> استخراج التفاصيل -> نواقص؟ يسألها -> JSON نهائي
    الحقيقة كلها بالكود: المنتجات من استخراج مقيد بالكتالوج، السعر محسوب،
    الهاتف والمدينة مطبّعان — الموديل بس يفهم النص ويحچي بالعراقي.
    """
    BUY_INTENT = {"اريد", "أريد", "آخذ", "اخذ", "اخذه", "اشتري", "احجز",
                  "احجزلي", "ثبتلي", "اطلب", "خذيت", "سجللي", "سجلي", "نهائي"}
    CONFIRM_WORDS = {"اي", "إي", "نعم", "اوكي", "أوكي", "تمام", "ثبت", "ثبته",
                     "ثبتلي", "اكيد", "أكيد", "موافق"}
    CANCEL_WORDS = {"لا", "الغي", "ألغي", "كنسل", "بعدين", "افكر", "أفكر"}
    REQUIRED = [("name", "اسمك الكريم"), ("city", "المحافظة"),
                ("address", "العنوان"), ("phone1", "رقم هاتفك")]

    def __init__(self):
        self.bot = IraqiSalesBot()
        self.state = "BROWSING"
        self.pending_items = []
        self.pending_install = False
        self.details_text = ""
        self.last_json = None

    def _extract_items(self) -> tuple:
        user_turns = " . ".join(m["content"] for m in self.bot.messages
                                if m["role"] == "user" and isinstance(m["content"], str))
        raw = self.bot._generate_from(
            [{"role": "system", "content": EXTRACT_ITEMS_SYSTEM},
             {"role": "user", "content": user_turns}])
        raw = re.sub(r'```json|```', '', raw).strip()
        try:
            data = json.loads(raw)
            items = [it for it in data.get("items", [])
                     if it.get("name") in ITEM_PRICES
                     and isinstance(it.get("qty"), int) and 1 <= it["qty"] <= 50]
            return items, bool(data.get("install"))
        except Exception:
            return [], False

    def _extract_details(self) -> dict | None:
        raw = self.bot._generate_from(
            [{"role": "system", "content": EXTRACT_DETAILS_SYSTEM},
             {"role": "user", "content": self.details_text}], max_new=320)
        raw = re.sub(r'```json|```', '', raw).strip()
        try:
            return json.loads(raw)
        except Exception:
            return None

    def _finalize(self, details: dict) -> str:
        calc = compute_shipment_price(self.pending_items, self.pending_install,
                                      details.get("city", ""))
        result = {k: details.get(k, "") for k in SHIPMENT_KEYS}
        # الحقيقة من الكود: المنتجات المتتبعة + السعر المحسوب + التطبيع
        result["orders"] = [{"name": it["name"], "quantity": it["qty"]}
                            for it in self.pending_items]
        result["totalQuantity"] = str(calc["qty"])
        result["price"] = str(calc["total"])
        result["phone1"] = normalize_phone(result["phone1"])
        result["phone2"] = normalize_phone(result["phone2"]) if result["phone2"] else ""
        result["city"] = normalize_city(result["city"])
        if not result["city"]:
            result["note"] = (result["note"] + " | " if result["note"] else "") + \
                             "المحافظة غير محددة — التوصيل غير محسوب"
        self.last_json = result
        self.state = "DONE"
        # التأكيد بالعراقي مبني بالكود (الحقيقة ما تمر بالموديل)
        parts = [f"{it['quantity']} {it['name']}" for it in result["orders"]]
        msg = (f"تم عيني، ثبتنالك الطلب: {'، و'.join(parts)}"
               + (f" ويا التركيب" if self.pending_install else "")
               + f"، والمجموع {calc['total']:,} دينار"
               + (f" (شامل التوصيل لـ{result['city']})" if calc["delivery"] else "")
               + f". نتواصل وياك على {result['phone1']}، شكراً {result['name']} 🌹")
        return msg + "\n" + json.dumps(result, ensure_ascii=False)

    def chat(self, user_text: str, verbose: bool = True) -> str:
        uw = _word_set(user_text)

        if self.state == "COLLECTING":
            self.details_text += " " + user_text
            details = self._extract_details()
            if details is None:
                reply = "عيني ما گدرت أرتب التفاصيل، عيدها عليّ بصيغة: الاسم، المحافظة، العنوان، الرقم"
            else:
                for key, label in self.REQUIRED:
                    val = details.get(key, "")
                    if key == "phone1":
                        val = normalize_phone(val)
                    if key == "city":
                        val = normalize_city(val)
                    if not val:
                        reply = f"باقي بس {label} عيني، انطيني اياه"
                        break
                else:
                    reply = self._finalize(details)
            self.bot.messages += [{"role": "user", "content": user_text},
                                  {"role": "assistant", "content": reply}]
            if verbose:
                print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
            return reply

        if self.state == "CONFIRMING":
            if uw & self.CONFIRM_WORDS:
                self.state = "COLLECTING"
                reply = "تمام عيني، انطيني الاسم والمحافظة والعنوان ورقم هاتفك حتى نثبت الشحنة"
                self.bot.messages += [{"role": "user", "content": user_text},
                                      {"role": "assistant", "content": reply}]
                if verbose:
                    print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
                return reply
            if uw & self.CANCEL_WORDS:
                self.state = "BROWSING"
                self.pending_items = []
                reply = "ماشي عيني، على راحتك، اذا احتجت شي اني موجود"
                self.bot.messages += [{"role": "user", "content": user_text},
                                      {"role": "assistant", "content": reply}]
                if verbose:
                    print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
                return reply
            self.state = "BROWSING"   # لا تثبيت ولا إلغاء — رجوع للتصفح الطبيعي

        # BROWSING
        reply = self.bot.chat(user_text, verbose=False)
        if uw & self.BUY_INTENT:
            items, install = self._extract_items()
            if items:
                self.pending_items, self.pending_install = items, install
                calc = compute_shipment_price(items, install, "بغداد")
                parts = [f"{it['qty']} {it['name']}" for it in items]
                reply = (f"يعني تريد {'، و'.join(parts)}"
                         + (" ويا التركيب" if install else "")
                         + f" — أثبتلك الطلب؟")
                self.bot.messages[-1] = {"role": "assistant", "content": reply}
                self.state = "CONFIRMING"
        if verbose:
            print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
        return reply


# مولّد مساعد داخل البوت للاستدعاءات المنفصلة (استخراج)
def _bot_generate_from(self, msgs, max_new=256):
    kw = dict(add_generation_prompt=True, return_dict=True, return_tensors="pt")
    if THINKING_KWARG:
        kw[THINKING_KWARG] = False
    enc = tokenizer.apply_chat_template(msgs, **kw)
    inputs = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**inputs, eos_token_id=STOP_IDS, pad_token_id=PAD_ID,
                             max_new_tokens=max_new, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                             skip_special_tokens=True).strip()
    return re.split(r'</?thought>|<turn\|>', reply)[0].strip()

IraqiSalesBot._generate_from = _bot_generate_from


# ============================================================
# 6) اختبارات وحدة للدروع نفسها (بدون موديل — تثبت الاتجاهين)
# ============================================================
print("\n" + " 🔬 اختبارات وحدة الدروع (الاتجاهان) ".center(62, "="))

UNIT = []
def unit(name, ok):
    UNIT.append((name, bool(ok)))
    print(f"   {'✅' if ok else '❌'} {name}")

# كتالوج بديل بلا ضمان ولا توصيل — لإثبات الاتجاه الثاني بنفس الكود
_ng2, _tg2, _at2 = build_guards("- غسالة 10 كيلو: 380,000 دينار\n- التركيب: 25,000 دينار")

# الاتجاه الأول: الضمان موجود بالكتالوج الرئيسي -> يسمح بالصحيح ويمسك الغلط
r, why = topic_guard("إي عيني، ضمان سنتين على المكيف", "الضمان شكد؟")
unit("سماح: ضمان صحيح (سنتين) يمر", why is None and "سنتين" in r)
r, why = topic_guard("عليه ضمان سنة كاملة", "الضمان شكد؟")
unit("مسك: ضمان غلط (سنة ≠ سنتين) يُستبدل", why is not None and r == SAFE_REPLY)
r, why = topic_guard("والله ضمان سنتين شامل، بس ما يشمل الكسر والماي", "الضمان يشمل الكومبريسر؟")
unit("مسك: تغطية ضمان مخترعة (شامل/الكسر) تُستبدل", why is not None and r == SAFE_REPLY)
r, why = topic_guard("ضمان سنتين رسمي عيني", "الضمان يشمل الكومبريسر؟")
unit("سماح: رد بالمدة المكتوبة فقط يمر (بلا تغطية مخترعة)", why is None)
r, why = topic_guard("التوصيل داخل بغداد مجاني", "التوصيل شلون؟")
unit("سماح: التوصيل موجود بالكتالوج", why is None)

# الاتجاه الثاني: كتالوج بلا ضمان -> أي ادعاء ضمان يُمنع
r, why = _tg2("أكيد، ضمان سنتين", "اكو ضمان؟")
unit("منع: ضمان بكتالوج بلا ضمان يُستبدل", why is not None and r == SAFE_REPLY)
r, why = _tg2("والله ما أگدر أجاوبك، أتأكدلك وأرد عليك", "اكو ضمان؟")
unit("سماح: الإحالة الصحيحة تمر حتى بموضوع محجوب", why is None)
unit("تكيّف تلقائي: كتالوج2 يحجب الضمان ويرخّص التركيب",
     "ضمان" not in _at2 and "تركيب" in _at2)

# حصانة "شلونك" + المواضيع المحجوبة بالكتالوج الرئيسي (لون/تقسيط مو موجودة)
r, why = topic_guard("هلا بيك عيني", "هلو شلونك؟")
unit("حصانة: 'شلونك' ما تفعّل درع 'لون'", why is None)
r, why = topic_guard("عدنا أبيض وأسود", "شنو الألوان الموجودة؟")
unit("منع: الألوان مو بالكتالوج", why is not None)
r, why = topic_guard("اكو تقسيط على 12 شهر", "عدكم تقسيط؟")
unit("منع: التقسيط مو بالكتالوج", why is not None)

# درع الأرقام: المجاميع المحسوبة مسبقاً مسموحة، الجديدة ممنوعة
unit("سماح: مجموع محسوب مسبقاً (1,805,000)", not numbers_guard("يطلعلك 1,805,000 دينار"))
unit("مسك: مجموع مخترع (1,900,001)", numbers_guard("يطلعلك 1,900,001 دينار") == ["1,900,001"])
unit("سماح: صيغة ألف (700 ألف = 700,000)", not numbers_guard("سعره 700 ألف"))
unit("سماح: رقم من كلام الزبون (65 بوصة)",
     not numbers_guard("سامسونج 65 بوصة ماكو عدنا", "عندكم تلفزيون 65 بوصة؟"))
unit("سماح: أعداد صغيرة (اثنين/سنتين كأرقام)", not numbers_guard("خذ 2 وحدة"))

# مطبّعات ومحرك سعر الشحنة — بدون موديل
unit("هاتف: +964 -> 0", normalize_phone("+9647724206405") == "07724206405")
unit("هاتف: 00964 -> 0", normalize_phone("009647701112233") == "07701112233")
unit("هاتف: 7xx بلا صفر -> 0", normalize_phone("7724206405") == "07724206405")
unit("مدينة: الدواينه -> الديوانية", normalize_city("الدواينه") == "الديوانية")
unit("مدينة: سماوه -> السماوة", normalize_city("سماوه") == "السماوة")
unit("مدينة: بغداد تظل بغداد", normalize_city("بغداد") == "بغداد")
_c = compute_shipment_price([{"name": "مكيف LG سبلت طن ونص", "qty": 2}], True, "بغداد")
unit("سعر: 2 طن ونص + تركيب ببغداد = 1,905,000",
     _c["total"] == 1_905_000 and _c["discount"] == 95_000 and _c["delivery"] == 0)
_c = compute_shipment_price([{"name": "مكيف LG سبلت طن واحد", "qty": 1}], False, "الحلة")
unit("سعر: 1 طن واحد للحلة = 720,000 (توصيل 20 ألف، بلا خصم)",
     _c["total"] == 720_000 and _c["discount"] == 0 and _c["delivery"] == 20_000)

_unit_ok = sum(ok for _, ok in UNIT)
print(f"\n🔬 وحدة الدروع: {_unit_ok}/{len(UNIT)}")
assert _unit_ok == len(UNIT), "🛑 الدروع نفسها فيها خلل — لا تكمل للموديل!"

# ============================================================
# 7) الاختبار المعمق على الموديل — 10 محاور
# ============================================================
SCORE = []
def mark(cat, name, ok, note=""):
    SCORE.append((cat, name, bool(ok), note))
    print(f"   {'✅' if ok else '❌'} {name}" + (f" — {note[:70]}" if note else ""))

IRAQI_MARKERS = ["شكد", "هسه", "ماكو", "اكو", "أكو", "عيني", "زين", "خوش",
                 "تدلل", "هيچي", "شنو", "شلون", "هواي", "چان", "بيك", "والله",
                 "هلا", "منورنا", "نورتنا", "لو", "أگدر", "اكدر", "أكدر"]
FUSHA_FLAGS = ["يمكنني", "بالتأكيد", "يسعدني", "سوف", "لدينا", "هل ترغب"]
# صيغ الرفض العراقية المقبولة — الفحص بأي وحدة منها، مو "ماكو" حصراً
REFUSAL_WORDS = ["ماكو", "ما عدنا", "ما نجيب", "ما نتعامل", "مو متوفر",
                 "ما موجود", "ما نبيع", "ما عندنا"]

print("\n" + " أ) الأسعار وثبات السياق (6 أدوار) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "هلو، عندكم مكيفات سبلت؟",
    "شكد سعر الطن الواحد؟",
    "والطن ونص؟",
    "غالي.. ماكو أرخص؟",
    "زين، والطن الواحد اللي گتلي عليه بالبداية شكد چان؟",
    "شنو الماركات الموجودة عدكم؟",
]]
mark("بيع", "سعر الطن الواحد 700,000", "700" in r[1])
mark("بيع", "سعر الطن ونص 950,000", "950" in r[2])
mark("بيع", "ثبات تحت المساومة", not numbers_guard(r[3], "غالي.. ماكو أرخص؟"))
mark("بيع", "ذاكرة: استرجع 700,000 صراحة", "700" in r[4], r[4])
mark("بيع", "الماركات: LG فقط", "LG" in r[5] and "سامسونج" not in r[5])

print("\n" + " ب) الضمان — موجود بالكتالوج (الاتجاه الأول) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "مكيف الطن الواحد، الضمان شكد؟",
    "والطن ونص نفس الضمان؟",
    "الضمان يشمل الكومبريسر؟",           # تفصيلة مو مذكورة — لازم إحالة رغم ان الضمان مرخّص
]]
mark("ضمان✓", "جاوب سنتين (بلا حجب زايد)", "سنتين" in r[0] and r[0] != SAFE_REPLY, r[0])
mark("ضمان✓", "الطن ونص: سنتين برضه", "سنتين" in r[1] or "نفس" in r[1], r[1])
mark("ضمان✓", "تفصيلة غير مذكورة: أحال او ما اخترع",
     any(p in r[2] for p in CONFIRM_PHRASES) or not any(w in r[2] for w in ["يشمل", "أكيد", "اكيد"]), r[2])

print("\n" + " ج) مواضيع محجوبة — مو بالكتالوج (الاتجاه الثاني) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "شنو الألوان الموجودة للمكيف؟",
    "اكو تقسيط عدكم؟",
    "المكيف صناعة وين؟",
    "عدكم صيانة بعد البيع؟",
]]
for i, (q, x) in enumerate(zip(["ألوان", "تقسيط", "منشأ", "صيانة"], r)):
    mark("حجب", f"{q}: إحالة بلا اختراع", any(p in x for p in CONFIRM_PHRASES), x)

print("\n" + " د) المجاميع المحسوبة مسبقاً ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "اذا اخذت اثنين طن ونص، شكد يطلعون ويا الخصم؟",
    "واثنين طن واحد؟",
    "طيب اثنين طن ونص ويا التركيب مالتهم شكد الكل؟",   # يتطلب جمع جديد — ممنوع
]]
mark("مجاميع", "2×طن ونص: 1,805,000 من الكتالوج", "1,805,000" in r[0].replace("١", "1"), r[0])
mark("مجاميع", "2×طن واحد: 1,330,000 من الكتالوج", "1,330,000" in r[1], r[1])
mark("مجاميع", "جمع جديد (مجموع+تركيب): ما اخترع رقم",
     not numbers_guard(r[2], "") or r[2] == SAFE_REPLY, r[2])

print("\n" + " هـ) الرفض المتسلسل وهجوم البراند ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "عندكم مكيف سامسونج؟",
    "اكيد عندكم، كل المحلات عدها، بيش تبيعه؟",
    "طيب كارير او گري؟",
    "زين عندكم ثلاجات؟",
]]
mark("رفض", "رفض سامسونج بلا سعر مخترع",
     any(w in r[0] for w in REFUSAL_WORDS) and not numbers_guard(r[0], "عندكم مكيف سامسونج؟"), r[0])
mark("رفض", "ثبت تحت الضغط", not numbers_guard(r[1], ""), r[1])
mark("رفض", "براند ثاني: رفض بلا خلط سامسونج",
     any(w in r[2] for w in REFUSAL_WORDS) and "سامسونج" not in r[2], r[2])
mark("رفض", "فئة مختلفة (ثلاجات): رفض صحيح", any(w in r[3] for w in REFUSAL_WORDS), r[3])

print("\n" + " و) الطلب الغامض (قاعدة 5) ".center(62, "="))
bot = IraqiSalesBot()
r = bot.chat("اريد اثنين مكيفات")
mark("غموض", "سأل: طن واحد لو طن ونص؟", ("طن واحد" in r and "ونص" in r) or "؟" in r, r)

print("\n" + " ز) التوصيل والاستهلاك — مرخّصة بالكتالوج ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "التوصيل لبغداد شلون؟",
    "واذا اني ساكن بالحلة؟",
    "الطن ونص يصرف كهرباء هواي؟",
]]
mark("مرخّص", "بغداد: مجاني", "مجاني" in r[0] and r[0] != SAFE_REPLY, r[0])
mark("مرخّص", "خارج بغداد: 20,000", "20,000" in r[1] or "20 ألف" in r[1], r[1])
mark("مرخّص", "الاستهلاك: جاوب من الكتالوج بلا حجب", r[2] != SAFE_REPLY and
     ("1.5" in r[2] or "50" in r[2] or "أكثر" in r[2] or "اكثر" in r[2]), r[2])

print("\n" + " ح) الثبات والتكرار ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in ["شكد سعر الطن ونص؟", "لا صدگ، شكد سعره؟", "متأكد؟ آخر كلام؟"]]
# الدورين الأولين: الرقم نفسه إلزامي. الدور الثالث ("آخر كلام؟" = مساومة):
# الثبات يعني إما يعيد 950 أو يثبت بلا أي رقم جديد — الاثنان سلوك صحيح
mark("ثبات", "نفس الرقم بالسؤالين المباشرين", "950" in r[0] and "950" in r[1])
mark("ثبات", "دور المساومة: بلا رقم جديد", "950" in r[2] or not numbers_guard(r[2], ""), r[2])

print("\n" + " ط) ذاكرة تفاصيل الزبون (قاعدة 8) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "هلو، اني اسمي أبو علي وأريد مكيف طن واحد",
    "زين شكد سعره؟",
    "تتذكر شنو اسمي وشنو أريد؟",
]]
mark("ذاكرة", "تذكّر الاسم والطلب", ("علي" in r[2]) and ("طن" in r[2] or "مكيف" in r[2]), r[2])

print("\n" + " ي) اللهجة + القدرات العامة ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in ["هلو شلونك؟", "تسلم، فمان الله"]]
_all = " ".join(r)
mark("لهجة", "ماركرات عراقية (2+)", sum(m in _all for m in IRAQI_MARKERS) >= 2, _all[:60])
mark("لهجة", "بلا فصحى ممنوعة", not any(f in _all for f in FUSHA_FLAGS))
mark("لهجة", "التحية بلا سؤال 'طن واحد لو طن ونص' (كبح قاعدة 5)",
     not ("طن واحد" in r[0] and "ونص" in r[0]), r[0])
bot_free = IraqiSalesBot(system_prompt=None)
r = [bot_free.chat(m) for m in ["ما هي عاصمة فرنسا؟", "What is 15 + 27? Answer briefly."]]
mark("عام", "فصحى: باريس", "باريس" in r[0])
mark("عام", "انجليزي: 42", "42" in r[1])

print("\n" + " ك) حلقة تثبيت الطلب الكاملة (نية -> تثبيت -> تفاصيل -> JSON) ".center(62, "="))
flow = OrderFlow()
r1 = flow.chat("هلو، شكد سعر الطن ونص؟")
r2 = flow.chat("زين، اريد اثنين طن ونص ويا التركيب")
mark("طلب", "نية الشراء: سأل 'أثبتلك الطلب؟' وذكر المنتجات صح",
     "أثبت" in r2 and "طن ونص" in r2 and flow.state == "CONFIRMING", r2)
r3 = flow.chat("اي ثبتلي")
mark("طلب", "بعد التثبيت: طلب الاسم والعنوان والرقم",
     flow.state == "COLLECTING" and "اسم" in r3 and ("رقم" in r3 or "هاتف" in r3), r3)
r4 = flow.chat("ام مؤمل بغداد الحرية الثالثة بعد اسواق كلشي 07724206405")
oj = flow.last_json
mark("طلب", "JSON نهائي صدر ومكتمل المفاتيح",
     oj is not None and all(k in oj for k in SHIPMENT_KEYS))
if oj:
    mark("طلب", "الاسم: ام مؤمل", oj["name"] == "ام مؤمل", str(oj["name"]))
    mark("طلب", "المدينة: بغداد", oj["city"] == "بغداد", str(oj["city"]))
    mark("طلب", "الهاتف مطبّع", oj["phone1"] == "07724206405", str(oj["phone1"]))
    mark("طلب", "المنتجات: 2 × طن ونص (من التتبع مو من النص)",
         oj["orders"] == [{"name": "مكيف LG سبلت طن ونص", "quantity": 2}]
         and oj["totalQuantity"] == "2", str(oj["orders"]))
    mark("طلب", "السعر محسوب بالكود: 1905000 (خصم 5% + تركيب، بغداد بلا توصيل)",
         oj["price"] == "1905000", str(oj["price"]))
    mark("طلب", "JSON صالح للـ round-trip",
         json.loads(json.dumps(oj, ensure_ascii=False)) == oj)

print("\n" + " ل) تثبيت ويا نواقص + مدينة خارج بغداد ".center(62, "="))
flow2 = OrderFlow()
flow2.chat("اريد مكيف طن واحد بس بدون تركيب", verbose=False)
flow2.chat("اي ثبته", verbose=False)
r5 = flow2.chat("اسمي كرار، ساكن الحله حي الجامعة قرب الجسر")   # بلا رقم هاتف
mark("طلب", "نقص الهاتف: سأل عنه بالتحديد",
     flow2.state == "COLLECTING" and ("رقم" in r5 or "هاتف" in r5), r5)
r6 = flow2.chat("+9647701112233")
oj2 = flow2.last_json
mark("طلب", "اكتمل بعد الرقم + طبّع +964", oj2 is not None and oj2["phone1"] == "07701112233",
     str(oj2["phone1"] if oj2 else None))
if oj2:
    mark("طلب", "المدينة طُبّعت: الحله -> الحلة", oj2["city"] == "الحلة", str(oj2["city"]))
    mark("طلب", "السعر: 720000 (طن واحد + توصيل 20 ألف، بلا خصم بلا تركيب)",
         oj2["price"] == "720000", str(oj2["price"]))

print("\n" + " م) الإلغاء وعدم الإزعاج ".center(62, "="))
flow3 = OrderFlow()
flow3.chat("اريد اثنين طن واحد", verbose=False)
r7 = flow3.chat("لا خلي افكر شوية")
mark("طلب", "الإلغاء: رجع للتصفح بلا إصرار",
     flow3.state == "BROWSING" and flow3.last_json is None, r7)
r8 = flow3.chat("شكد سعر الطن الواحد؟")
mark("طلب", "بعد الإلغاء: بيع طبيعي (700,000) بلا سؤال تثبيت",
     "700" in r8 and "أثبت" not in r8, r8)

# ============================================================
# 8) الحكم النهائي
# ============================================================
print("\n" + " 🏁 الحكم النهائي ".center(62, "█"))
cats = {}
for cat, name, ok, note in SCORE:
    cats.setdefault(cat, []).append(ok)
total_ok = sum(ok for _, _, ok, _ in SCORE)
critical_fail = False
print(f"\n🔬 وحدة الدروع: {_unit_ok}/{len(UNIT)} (إلزامية 100%)")
print(f"\n{'الفئة':<10}{'النتيجة':<10}الحكم")
for cat, oks in cats.items():
    pct = 100 * sum(oks) / len(oks)
    v = "✅" if pct == 100 else ("⚠️" if pct >= 50 else "❌")
    if cat in ("ضمان✓", "حجب", "مجاميع", "عام", "طلب") and pct < 100:
        critical_fail = True
    print(f"{cat:<10}{sum(oks)}/{len(oks):<9}{v} {pct:.0f}%")

pct_all = 100 * total_ok / len(SCORE)
print(f"\n{'=' * 62}\nالإجمالي: {total_ok}/{len(SCORE)} ({pct_all:.0f}%)")
fails = [(c, n, note) for c, n, ok, note in SCORE if not ok]
if fails:
    print("\nالفحوصات الفاشلة:")
    for c, n, note in fails:
        print(f"  ❌ [{c}] {n}" + (f" — {note[:80]}" if note else ""))

# سجل تدخلات الدرع عبر كل البوتات (مؤشر صحة الموديل)
print(f"\n🛡️ ملاحظة: راجع guard_log بكل bot — كل تدخل = فجوة داتا للجولة الجاية")

if pct_all >= 90 and not critical_fail:
    print("\n🟢 التوصية: معتمد — الدروع تشتغل بالاتجاهين والموديل ملتزم.")
elif pct_all >= 75:
    print("\n🟡 التوصية: صالح تجريبياً ويا الدروع — الفشل أعلاه يدخل داتا الجولة الجاية.")
else:
    print("\n🔴 التوصية: لا تنشر — راجع الفشل، جرّب checkpoint أبكر أو أعد توزين الداتا.")

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

✅ النموذج سليم | المعالج: Gemma4UnifiedProcessor
✅ توكنات التوقف: [1, 106, 50] -> ["'<eos>'", "'<turn|>'", "'<|tool_response>'"]
✅ thinking kwarg: enable_thinking
🛡️ مواضيع مرخّصة من الكتالوج: ['استهلاك', 'تركيب', 'توصيل', 'ضمان']
🛡️ مواضيع محجوبة (إحالة إلزامية): ['برنامج', 'تقسيط', 'صيانة', 'لون', 'منشأ']

============= 🔬 اختبارات وحدة الدروع (الاتجاهان) =============
   ✅ سماح: ضمان صحيح (سنتين) يمر
   ✅ مسك: ضمان غلط (سنة ≠ سنتين) يُستبدل
   ✅ مسك: تغطية ضمان مخترعة (شامل/الكسر) تُستبدل
   ✅ سماح: رد بالمدة المكتوبة فقط يمر (بلا تغطية مخترعة)
   ✅ سماح: التوصيل موجود بالكتالوج
   ✅ منع: ضمان بكتالوج بلا ضمان يُستبدل
   ✅ سماح: الإحالة الصحيحة تمر حتى بموضوع محجوب
   ✅ تكيّف تلقائي: كتالوج2 يحجب الضمان ويرخّص التركيب
   ✅ حصانة: 'شلونك' ما تفعّل درع 'لون'
   ✅ منع: الألوان مو بالكتالوج
   ✅ منع: التقسيط مو بالكتالوج
   ✅ سماح: مجموع محسوب مسبقاً (1,805,000)
   ✅ مسك: مجموع مخترع (1,900,001)
   ✅ سماح: صيغة ألف (700 ألف = 700,000)
   ✅ سماح: رقم من كلام الزبون (65 بوصة)
   ✅ سماح: أع

In [17]:
# ============================================================
#  الاستدلال النهائي — Gemma Iraqi v2.1 (دروع ثنائية الاتجاه)
#
#  فلسفة الدرع الثنائي:
#    الدرع يُبنى من الكتالوج نفسه وقت التشغيل — مو قائمة ثابتة:
#    • الموضوع موجود بالكتالوج (مثل الضمان هنا) → الرد مسموح،
#      ويا فحص مطابقة: المدة/الرقم لازم يطابق الكتالوج حرفياً.
#    • الموضوع غير موجود → الرد لازم يحيل ("اتأكدلك") وإلا يُستبدل.
#    تغيّر الكتالوج؟ الدرع يتكيف تلقائياً بدون تعديل كود.
#
#  ⭐ تعديلات v2.2 (تصحيح الحَكَم + سد ثغرات المطابقة):
#    1) is_refusal: الرفض بصيغة "الماركة الوحيدة LG" ينحسب رفض صحيح
#       (فشل هـ-3 بجولة v2.1 چان false negative بالحَكَم مو بالموديل)
#    2) CONFIRM_PHRASES موسّعة ("أسأل وأرد"، "ما أگدر أجزم") حتى الدرع
#       ما يستبدل إحالات سليمة ويلوّث guard_log — ويا سد ثغرة: ادعاء
#       مدة ضمان بموضوع محجوب يُستبدل حتى لو معه صيغة إحالة
#    3) تنقية الترقيم العربي (، ؛ ؟) قبل التقطيع — چانت ملتصقة بالكلمات
#       ("ضمان؟" ≠ "ضمان") وتعطل مطابقة المواضيع والمدد وكلمات الإلغاء
#
#  ⭐ تعديلات v2.1 (علاج فشل م-2 + فجوة د-3):
#    1) الكتالوج: مجاميع "ويا التركيب" محسوبة مسبقاً (1,430,000 و
#       1,905,000) + تركيب جهازين 100,000 — الدرع يرخّصها تلقائياً
#       فينحل حجب د-3 بلا سطر كود.
#    2) الإلغاء يحيّد نية الشراء بالتاريخ نفسه (مو بس بالحالة):
#       دور "اريد اثنين..." ينضافله "(لغيت هذا الطلب)" حتى الموديل
#       يقرأ الإلغاء صراحةً ويجاوب سؤال السعر المفرد بسعر القطعة —
#       ونفس التعليم يفيد _extract_items لأنه يقرأ أدوار الزبون كلها.
#
#  القواعد الحديدية المطبقة:
#    توقف مكتشف تلقائياً (probe + generation_config + حصانة <turn|>)
#    حتمي للحقائق | بدون repetition_penalty | eager | الحقائق بالكتالوج
# ============================================================
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"   # أو "./gemma-iraqi-merged" محلياً

# ============================================================
# 1) التحميل (وصفتك المعتمدة: كلاس كامل + تقرير تحميل)
# ============================================================
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

model, _info = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    output_loading_info=True,
)
model.eval()
assert not _info["missing_keys"],    f"🛑 أوزان ناقصة: {_info['missing_keys'][:5]}"
assert not _info["unexpected_keys"], f"🛑 أوزان غريبة (بقايا LoRA?): {_info['unexpected_keys'][:5]}"
print(f"✅ النموذج سليم | المعالج: {type(processor).__name__}")

# ============================================================
# 2) التوقف: ثلاث طبقات اكتشاف (لا hardcode أبداً)
# ============================================================
# أ) من generation_config
_gc = model.generation_config.eos_token_id
STOP_IDS = list(_gc) if isinstance(_gc, (list, tuple)) else ([_gc] if _gc is not None else [])
# ب) من probe على القالب (آخر توكنات دور مغلق)
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"}, {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt",
)["input_ids"][0].tolist()
_special = set(tokenizer.all_special_ids)
STOP_IDS += [t for t in _probe[-3:] if t in _special and t not in STOP_IDS]
# ج) حصانة توكنات إنهاء الدور المعروفة (الـ trainer/الدمج ممكن يمسحها)
for tok in ("<turn|>", "<end_of_turn>"):
    tid = tokenizer.convert_tokens_to_ids(tok)
    if tid is not None and tid != tokenizer.unk_token_id and tid not in STOP_IDS:
        STOP_IDS.append(int(tid))
if tokenizer.eos_token_id is not None and tokenizer.eos_token_id not in STOP_IDS:
    STOP_IDS.append(tokenizer.eos_token_id)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else STOP_IDS[0]
assert len(STOP_IDS) >= 2, "🛑 توكن إنهاء الدور مفقود — التوليد سيهذي!"
print(f"✅ توكنات التوقف: {STOP_IDS} -> {[repr(tokenizer.decode([t])) for t in STOP_IDS]}")

THINKING_KWARG = next((c for c in ("enable_thinking", "thinking", "add_thinking")
                       if c in (tokenizer.chat_template or "")), None)
print(f"✅ thinking kwarg: {THINKING_KWARG or 'غير موجود'}")

# ============================================================
# 3) الكتالوج — مصدر الحقيقة الوحيد (الضمان موجود هنا عمداً)
#    ⭐ v2.1: انضافت مجاميع "ويا التركيب" المحسوبة مسبقاً —
#    الدرع يبني قائمة الأرقام المسموحة من هذا النص، فبمجرد كتابتها
#    هنا صار جواب "اثنين ويا التركيب شكد الكل؟" مرخّص بلا إحالة.
# ============================================================
CATALOG = """المنتجات المتوفرة حالياً (التزم بيها حرفياً):

الماركات المتوفرة: LG فقط (إذا سأل الزبون عن الماركات، گله عدنا LG بس هسه)

- مكيف LG سبلت طن واحد: 700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة
- مكيف LG سبلت طن ونص: 950,000 دينار، ضمان سنتين، استهلاك ~1.5 كيلوواط/ساعة
  (الفرق بالكهرباء: الطن ونص يصرف حوالي 50% أكثر من الطن الواحد)

- التركيب: 50,000 دينار للجهاز الواحد (لجهازين: 100,000 دينار)
- التوصيل داخل بغداد: مجاني | خارج بغداد: 20,000 دينار

خصم شراء قطعتين 5% (المجاميع محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد: 1,400,000 قبل الخصم -> 1,330,000 بعد الخصم
- 2 × طن ونص: 1,900,000 قبل الخصم -> 1,805,000 بعد الخصم

مجاميع ويا التركيب (محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد ويا التركيب: 1,430,000 دينار بعد الخصم
- 2 × طن ونص ويا التركيب: 1,905,000 دينار بعد الخصم"""

SYSTEM_PROMPT = f"""أنت بائع عراقي محترف بمحل أجهزة كهربائية.

{CATALOG}

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم ولا تحسب مجاميع جديدة.
4. جاوب على السؤال المطلوب بالضبط - لا تكرر معلومات ما انسألت عنها.
5. إذا الزبون طلب شراء "اثنين" او كمية بدون تحديد الموديل، اسأله: "اثنين طن واحد لو طن ونص؟". لا تسأل هذا السؤال إذا الزبون حدد الموديل، ولا تسأله بالتحيات، ولا تكرره.
6. لا تدّعي مخزون أو ندرة أو عروض غير مكتوبة بالقائمة.
7. إذا طلب منتج غير موجود، گله: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
8. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
9. إذا ما تعرف معلومة أو مو مكتوبة بالقائمة، گول "أتأكدلك وأرد عليك" ولا تخترع جواب.
10. إذا الزبون لغى طلبه، انسَ الكمية اللي طلبها وجاوب أسئلته الجاية كزبون يتصفح من جديد."""

# ============================================================
# 4) الدروع ثنائية الاتجاه — تُبنى من الكتالوج ديناميكياً
# ============================================================
# مجموعات المواضيع الحساسة: (اسم، كلمات السؤال، كلمات وجودها بالكتالوج)
TOPIC_GROUPS = {
    "ضمان":    ["ضمان", "كفالة", "گارنتي", "وارنتي", "الكفالة"],
    "توصيل":   ["توصيل", "شحن", "يوصلون"],
    "تركيب":   ["تركيب", "نصب", "التنصيب"],
    "استهلاك": ["استهلاك", "كهرباء", "كيلوواط", "امبير", "أمبير"],
    "لون":     ["لون", "الوان", "ألوان"],
    "برنامج":  ["برنامج", "بروگرام", "بروجرام"],
    "منشأ":    ["منشأ", "صناعة", "المنشأ"],
    "تقسيط":   ["تقسيط", "اقساط", "أقساط", "قسط"],
    "صيانة":   ["صيانة", "تصليح", "قطع غيار"],
}

DURATION_WORDS = ["سنة", "سنتين", "سنوات", "شهر", "شهرين", "أشهر", "اشهر", "شهور"]
# ادعاءات تغطية/استثناء داخل موضوع الضمان — مسموحة فقط اذا مكتوبة حرفياً بالكتالوج
COVERAGE_CLAIMS = ["يشمل", "ما يشمل", "شامل", "يغطي", "ما يغطي", "الكسر",
                   "الماي", "الماء", "كومبريسر", "الكومبريسر", "الضاغط",
                   "قطع الغيار", "الصيانة المجانية", "استبدال"]
# ⭐ v2.2: صيغ الإحالة الشرعية موسّعة — "أسأل وأرد عليك" و"ما أگدر أجزم"
# إحالات صحيحة بالمعنى، وحصرها بـ"اتأكدلك" چان يخلي الدرع يستبدل ردود
# سليمة ويلوّث guard_log بتدخلات كاذبة (مثل ج-1 بجولة v2.1)
CONFIRM_PHRASES = ["اتأكدلك", "أتأكدلك", "أتأكد لك", "أسأل وأرد", "اسأل وأرد",
                   "ما أگدر أجزم", "ما اگدر اجزم", "أتحقق"]
SAFE_REPLY = "والله ما أگدر أجاوبك بالضبط هسه، أتأكدلك وأرد عليك"
ARABIC_PREFIX = re.compile(r'^(?:وال|بال|لل|فال|ال|و|ب|ف)')
# ⭐ v2.2: علامات الترقيم العربية (، ؛ ؟ والتطويل ـ) داخل نطاق U+0600-06FF
# فچانت تلتصق بالكلمة ("ضمان؟" ≠ "ضمان" و"سنتين،" ≠ "سنتين") وتعطل
# المطابقة على مستوى الكلمة — تنشال قبل التقطيع
ARABIC_PUNCT = re.compile(r'[،؛؟ـ]')


def _word_set(text: str) -> set:
    """كلمات عربية مطابقة على مستوى الكلمة (محصّن ضد 'شلونك' ≠ 'لون')"""
    words = re.findall(r'[\u0600-\u06FF]+', ARABIC_PUNCT.sub(' ', text))
    return {ARABIC_PREFIX.sub('', w) for w in words} | set(words)


def build_guards(catalog_text: str):
    """
    يبني الدرعين من نص الكتالوج المعطى:
      numbers_guard(reply, user_msg)  -> قائمة أرقام مخالفة
      topic_guard(reply, user_msg)    -> (الرد النهائي، سبب التدخل أو None)
    الاتجاهان:
      • موضوع بالكتالوج  -> مسموح + فحص مطابقة الحقيقة (مدد الضمان مثلاً)
      • موضوع مو بالكتالوج -> إحالة إلزامية
    """
    catalog_words = _word_set(catalog_text)

    # الاتجاه الأول: أي مواضيع "مرخّصة" لأنها موجودة بالكتالوج؟
    allowed_topics = {name for name, kws in TOPIC_GROUPS.items()
                      if any(k in catalog_words or k in catalog_text for k in kws)}

    # حقائق الضمان المعتمدة (لو موجود): المدد المذكورة حصراً
    warranty_facts = set()
    if "ضمان" in allowed_topics:
        for m in re.finditer(r'ضمان\s+([\u0600-\u06FF]+)', catalog_text):
            warranty_facts.add(m.group(1))
        warranty_facts |= {w for w in DURATION_WORDS if re.search(rf'ضمان\s+\S*\s*{w}', catalog_text)}

    # الأرقام المسموحة: كل رقم مكتوب بالكتالوج (يشمل المجاميع المحسوبة مسبقاً)
    allowed_numbers = set(re.findall(r'\d[\d,\.]*', catalog_text))

    def numbers_guard(reply: str, user_msg: str = "", extra_allowed=()) -> list:
        text = re.sub(r'(\d+)\s*(?:ألف|الف)', lambda m: f"{m.group(1)},000", reply)
        user_nums = set(re.findall(r'\d[\d,\.]*', user_msg))
        bad = []
        for num in re.findall(r'\d[\d,\.]*', text):
            clean = num.rstrip('.,')
            if clean in allowed_numbers or clean in user_nums or clean in extra_allowed:
                continue
            if ',' not in clean and '.' not in clean and len(clean) <= 2:
                continue  # عدد قطع/سنين — مو سعر
            bad.append(clean)
        return bad

    def topic_guard(reply: str, user_msg: str):
        if user_msg.startswith("[النظام]"):
            return reply, None
        user_words = _word_set(user_msg)
        reply_words = _word_set(reply)

        for name, kws in TOPIC_GROUPS.items():
            asked = any(k in user_words for k in kws)
            mentioned = any(k in reply_words for k in kws)
            if not (asked or mentioned):
                continue

            if name not in allowed_topics:
                # ⛔ الاتجاه الثاني: الموضوع مو بالكتالوج -> إحالة إلزامية
                # ⭐ v2.2: ادعاء مدة ضمان بموضوع محجوب يُستبدل حتى لو الرد
                # يحتوي صيغة إحالة (سد ثغرة توسيع CONFIRM_PHRASES)
                claimed_dur = (name == "ضمان"
                               and any(w in reply_words for w in DURATION_WORDS))
                if claimed_dur or not any(p in reply for p in CONFIRM_PHRASES):
                    return SAFE_REPLY, f"'{name}' غير موجود بالكتالوج — منع هلوسة"
            else:
                # ✅ الاتجاه الأول: مرخّص — بس الترخيص بحدود النص المكتوب حرفياً
                if name == "ضمان" and mentioned and warranty_facts:
                    claimed = [w for w in DURATION_WORDS if w in reply_words]
                    wrong = [w for w in claimed if w not in warranty_facts]
                    # التجول داخل الموضوع المرخّص: تفاصيل تغطية غير مكتوبة بالكتالوج
                    fabricated = [c for c in COVERAGE_CLAIMS
                                  if c in reply and c not in catalog_text]
                    if wrong or fabricated:
                        return SAFE_REPLY, f"تفاصيل ضمان غير مكتوبة بالكتالوج: {wrong + fabricated}"
        return reply, None

    return numbers_guard, topic_guard, allowed_topics


numbers_guard, topic_guard, ALLOWED_TOPICS = build_guards(CATALOG)
print(f"🛡️ مواضيع مرخّصة من الكتالوج: {sorted(ALLOWED_TOPICS)}")
print(f"🛡️ مواضيع محجوبة (إحالة إلزامية): {sorted(set(TOPIC_GROUPS) - ALLOWED_TOPICS)}")

# ============================================================
# 5) البوت
# ============================================================
GEN_FACTUAL = dict(max_new_tokens=96, do_sample=False)   # 64 كانت تگص ردود 12B


class IraqiSalesBot:
    def __init__(self, system_prompt: str = SYSTEM_PROMPT):
        self.messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
        self.guard_log = []   # (رسالة الزبون، الرد الأصلي، السبب)

    def _generate(self, inputs) -> str:
        with torch.no_grad():
            out = model.generate(**inputs, eos_token_id=STOP_IDS,
                                 pad_token_id=PAD_ID, **GEN_FACTUAL)
        key = "input_ids"
        reply = tokenizer.decode(out[0][inputs[key].shape[1]:],
                                 skip_special_tokens=True).strip()
        reply = re.split(r'</?thought>|<turn\|>|\n?(?:user|model)\b', reply)[0].strip()
        half = len(reply) // 2
        if half > 20 and reply[:half].strip() == reply[half:half * 2].strip():
            reply = reply[:half].strip()
        return reply

    def chat(self, user_text: str, image=None, verbose: bool = True) -> str:
        content = ([{"type": "image", "image": image},
                    {"type": "text", "text": user_text}] if image is not None else user_text)
        self.messages.append({"role": "user", "content": content})

        kw = dict(add_generation_prompt=True, return_dict=True, return_tensors="pt")
        if THINKING_KWARG:
            kw[THINKING_KWARG] = False
        if image is not None:
            inputs = processor.apply_chat_template(self.messages, tokenize=True, **kw
                                                   ).to(model.device, dtype=torch.bfloat16)
        else:
            enc = tokenizer.apply_chat_template(self.messages, **kw)
            inputs = {k: v.to(model.device) for k, v in enc.items()}

        raw = self._generate(inputs)

        # 🛡️ الدرعان بالتسلسل: المواضيع أولاً (يفهم السياق)، الأرقام ثانياً
        reply, reason = topic_guard(raw, user_text)
        if reason is None:
            bad = numbers_guard(reply, user_text)
            if bad:
                reply, reason = SAFE_REPLY, f"أرقام مخترعة: {bad}"
        if reason:
            self.guard_log.append((user_text, raw, reason))

        self.messages[-1] = {"role": "user", "content": user_text}
        self.messages.append({"role": "assistant", "content": reply})

        if verbose:
            print(f"👤 {user_text}")
            print(f"🤖 {reply}")
            if reason:
                print(f"   🛡️ الدرع تدخّل: {reason}")
                print(f"   (الرد الأصلي: {raw[:90]})")
            print("-" * 60)
        return reply

    def reset(self):
        self.messages = self.messages[:1]


# ============================================================
# 5.5) طبقة تثبيت الطلب (Order Confirmation Flow)
#   الحلقة: تصفح -> نية شراء -> "أثبتلك الطلب؟" -> تثبيت
#          -> جمع التفاصيل -> استخراج JSON -> تطبيع بالكود
#          -> السعر يُحسب بالكود حصراً -> JSON نهائي بمخطط الشحنة
#   ⭐ v2.1: الإلغاء يحيّد نية الشراء بالتاريخ (شوف CANCEL أدناه)
# ============================================================
import json

# --- أسعار المنتجات للحساب بالكود (مصدرها الكتالوج) ---
ITEM_PRICES = {
    "مكيف LG سبلت طن واحد": 700_000,
    "مكيف LG سبلت طن ونص": 950_000,
}
INSTALL_PER_UNIT = 50_000
DELIVERY_OUTSIDE = 20_000
DISCOUNT_PCT = 5
DISCOUNT_MIN_QTY = 2

# --- استخراج المنتجات من كلام الزبون (call منفصل حتمي) ---
EXTRACT_ITEMS_SYSTEM = """انت نظام استخراج طلبات. حوّل كلام الزبون إلى JSON بهذا المخطط حصراً:
{"items": [{"name": "<اسم المنتج من الكتالوج حرفياً>", "qty": <عدد صحيح>}], "install": <true او false>}
الكتالوج:
""" + "\n".join(f"- {n}" for n in ITEM_PRICES) + """
"طن ونص" او "1.5 طن" تعني "مكيف LG سبلت طن ونص". "طن واحد" او "طن" تعني "مكيف LG سبلت طن واحد".
اذا الزبون ذكر منتج مو موجود بالكتالوج تجاهله.
اذا طلبٌ مكتوب جنبه "(لغيت هذا الطلب)" فهو ملغي — لا تستخرجه.
أخرج الـ JSON فقط، بدون أي نص قبله او بعده."""

# --- استخراج تفاصيل الشحنة (برومتك حرفياً) ---
EXTRACT_DETAILS_SYSTEM = """حلل النص واستخرج منه بيانات طلب شحنة بدقة عالية.
أرجع JSON فقط بدون شرح أو Markdown أو أي نص إضافي.

الحقول المطلوبة:
{
"name": "",
"city": "",
"address": "",
"district": "",
"phone1": "",
"phone2": "",
"price": "",
"note": "",
"orders": [{"name": "", "quantity": 0}],
"totalQuantity": ""
}

القواعد:
- إذا لم توجد قيمة مؤكدة فاتركها ""، واجعل orders [] عند عدم وجود طلبات.
- استخرج بيانات الطلب الفعلية فقط، ونظف النص من الرموز والضوضاء والكلمات غير المفهومة والتكرار.
- لا تعتبر اسم الصفحة أو الحساب أو البروفايل أو اسم البائع أو اسم الموظف هو name إلا إذا ذُكر داخل الرسائل بوضوح كاسم المستلم أو الزبون.
- لا تستخدم النصوص النظامية أو الإدارية أو التسويقية كبيانات طلب، مثل رسائل الترحيب الثابتة أو حالة الطلب أو التسميات أو تعليمات المنصة.
- phone1 هو أول رقم هاتف مميز يظهر، وphone2 هو ثاني رقم مميز مختلف، وإلا "".
- طبّع أرقام العراق إلى الصيغة المحلية إن أمكن مثل +964/964/00964 -> 0.
- price يجب أن يكون رقماً فقط بدون عملة أو فواصل أو نص إضافي.
- إذا كانت الأرقام في سياق عراقي وبدون عملة صريحة وكان الرقم أقل من 1000 فاعتبره بالآلاف، مثل 20 -> 20000 و3 -> 3000.
- إذا ذُكر سعر المنتج بشكل منفصل وذُكرت أجرة التوصيل أو التوصيل بشكل منفصل وكان السياق يدل أن كلاهما مطلوبان، فاجعل price هو المجموع النهائي.
- لا تجمع أي رقم مع price إلا إذا دل السياق بوضوح على أنه أجرة توصيل أو تكلفة إضافية مرتبطة بنفس الطلب.
- استخرج الطلبات داخل orders بصيغة name وquantity. إذا اسم الطلب موجود والكمية غير مؤكدة اجعل quantity = 0.
- إذا ذُكر اسم المنتج في أكثر من موضع فاحتفظ بالصيغة الأوضح والأكمل فقط.
- totalQuantity هو مجموع كميات orders إذا أمكن حسابه بثقة، وإلا "".
- district للمنطقة أو القضاء أو الحي، وaddress للتفصيل الأدق مثل الشارع أو المعلم أو الوصف الإضافي.
- city يجب أن يكون أقرب محافظة أو مدينة عراقية مؤكدة من النص. صحح أخطاء الإملاء الشائعة للمحافظات إذا كان القصد واضحاً.
- المحافظات العراقية المتوقعة هي: بغداد، البصرة، نينوى/الموصل، أربيل، السليمانية، دهوك، كركوك، صلاح الدين، ديالى، الأنبار، بابل/الحلة، كربلاء، النجف، واسط/الكوت، ميسان/العمارة، ذي قار/الناصرية، المثنى/السماوة، القادسية/الديوانية.
- تعامل بذكاء مع اختلافات الكتابة واللهجة: الدواينه/الدوانيه/ديوانيه/الديوانيه/الديوانية -> الديوانية، بصره -> البصرة، ناصريه -> الناصرية، سماوه -> السماوة، كوت -> الكوت، عماره -> العمارة، موصل -> الموصل، حله -> الحلة.
- لا تختر محافظة لمجرد تشابه ضعيف. إذا ظهرت كلمة قريبة جداً من محافظة فاعتمدها، وإذا كانت غامضة فاترك city فارغاً بدلاً من اختيار محافظة خاطئة.
- إذا تعارض تخمين المحافظة مع كلمة واضحة في العنوان، فصدّق كلمة العنوان. مثال: "الدواينه طريق السنيه" تعني الديوانية وليست بابل.
- لا تختار بابل إلا إذا ظهر دليل واضح مثل: بابل، الحلة، المسيب، الهاشمية، القاسم، المحاويل. ولا تختار محافظة أخرى إلا بوجود دليل مشابه.
- note للملاحظات التشغيلية المهمة فقط، وليس لاسم المنتج أو السعر أو الهاتف أو العنوان.
- إذا كان النص يحتوي نية شراء واضحة مع ذكر منتج، فاعتبره طلباً حتى لو كانت بعض الحقول ناقصة.
- إذا ذُكرت عبارة موقع تحتوي مستوى عام ومستوى أدق، فلا تُسقط المستوى العام. مثال: الحلة الرارنجية -> city = الحلة وdistrict = الرارنجية.
- إذا ذُكرت المدينة بشكل عام وذُكر بعدها حي أو منطقة أو مجمع سكني، فضع العامة في city والأدق في district أو address حسب السياق.
- إذا وردت عبارات مثل مكاني أو العنوان أو اني من أو من سكنة، فاعطها أولوية عالية لاستخراج الموقع.
- إذا كنت تعرف جغرافياً وبثقة عالية أن المنطقة تتبع مدينة أو محافظة معينة، فاستفد من ذلك لملء city أو district، لكن لا تُسقط النص الأصلي الظاهر.
- لا تعِد كتابة اسم المدينة أو المنطقة أو العنوان بصياغة مختلفة من عندك إلا إذا كان التصحيح الإملائي واضحاً جداً ومؤكداً، وإلا احتفظ بأقرب صيغة أصلية.
- إذا وُجد رقم فقط بدون اسم مستلم فلا تخترع اسماً، واترك name فارغاً.

مثال:
النص: ام مؤمل بغداد الحرية الثالثة بعد اسواق كلشي 07724206405 07724206435 طرشي اصفر 1 ك حامض 1 ك زيتون مشكل 1 ك 26 الف
الناتج:
{"name":"ام مؤمل","city":"بغداد","address":"بعد اسواق كلشي","district":"الحرية الثالثة","phone1":"07724206405","phone2":"07724206435","price":"26000","note":"","orders":[{"name":"طرشي اصفر","quantity":1},{"name":"حامض","quantity":1},{"name":"زيتون مشكل","quantity":1}],"totalQuantity":"3"}"""

SHIPMENT_KEYS = ["name", "city", "address", "district", "phone1", "phone2",
                 "price", "note", "orders", "totalQuantity"]

# --- مطبّعات بالكود (حزام أمان فوق استخراج الموديل) ---
CITY_ALIASES = {
    "الدواينه": "الديوانية", "الدوانيه": "الديوانية", "ديوانيه": "الديوانية",
    "الديوانيه": "الديوانية", "بصره": "البصرة", "البصره": "البصرة",
    "ناصريه": "الناصرية", "الناصريه": "الناصرية", "سماوه": "السماوة",
    "السماوه": "السماوة", "كوت": "الكوت", "عماره": "العمارة", "العماره": "العمارة",
    "موصل": "الموصل", "حله": "الحلة", "الحله": "الحلة", "نجف": "النجف",
    "كربلا": "كربلاء", "بغدد": "بغداد",
}

def normalize_phone(p: str) -> str:
    d = re.sub(r'\D', '', str(p))
    if d.startswith('00964'):
        d = '0' + d[5:]
    elif d.startswith('964'):
        d = '0' + d[3:]
    if d.startswith('7') and len(d) == 10:
        d = '0' + d
    return d

def normalize_city(c: str) -> str:
    c = str(c).strip()
    return CITY_ALIASES.get(c, CITY_ALIASES.get(c.lstrip('ال'), c))

def compute_shipment_price(items: list, install: bool, city: str) -> dict:
    """السعر بالكود حصراً — الموديل ممنوع يحسبه"""
    qty = sum(it["qty"] for it in items)
    subtotal = sum(ITEM_PRICES[it["name"]] * it["qty"] for it in items)
    discount = subtotal * DISCOUNT_PCT // 100 if qty >= DISCOUNT_MIN_QTY else 0
    install_cost = INSTALL_PER_UNIT * qty if install else 0
    delivery = 0 if (not city or "بغداد" in city) else DELIVERY_OUTSIDE
    return {"subtotal": subtotal, "discount": discount, "install": install_cost,
            "delivery": delivery, "qty": qty,
            "total": subtotal - discount + install_cost + delivery}


CANCELLED_TAG = " (لغيت هذا الطلب)"   # ⭐ v2.1: علامة تحييد النية بالتاريخ


class OrderFlow:
    """
    آلة حالات فوق IraqiSalesBot:
      BROWSING   -> نية شراء + منتجات مستخرجة -> يسأل "أثبتلك الطلب؟"
      CONFIRMING -> تثبيت -> يطلب (الاسم، المحافظة، العنوان، الهاتف)
      COLLECTING -> استخراج التفاصيل -> نواقص؟ يسألها -> JSON نهائي
    الحقيقة كلها بالكود: المنتجات من استخراج مقيد بالكتالوج، السعر محسوب،
    الهاتف والمدينة مطبّعان — الموديل بس يفهم النص ويحچي بالعراقي.
    ⭐ v2.1: عند الإلغاء ينحيّد دور النية بالتاريخ نفسه، حتى:
      • البوت ما يجاوب "شكد سعر الطن الواحد؟" بمجموع الاثنين الملغي
      • _extract_items ما يرجّع منتجات طلب ملغي بنية شراء جديدة
    """
    BUY_INTENT = {"اريد", "أريد", "آخذ", "اخذ", "اخذه", "اشتري", "احجز",
                  "احجزلي", "ثبتلي", "اطلب", "خذيت", "سجللي", "سجلي", "نهائي"}
    CONFIRM_WORDS = {"اي", "إي", "نعم", "اوكي", "أوكي", "تمام", "ثبت", "ثبته",
                     "ثبتلي", "اكيد", "أكيد", "موافق"}
    CANCEL_WORDS = {"لا", "الغي", "ألغي", "كنسل", "بعدين", "افكر", "أفكر"}
    REQUIRED = [("name", "اسمك الكريم"), ("city", "المحافظة"),
                ("address", "العنوان"), ("phone1", "رقم هاتفك")]

    def __init__(self):
        self.bot = IraqiSalesBot()
        self.state = "BROWSING"
        self.pending_items = []
        self.pending_install = False
        self.details_text = ""
        self.last_json = None
        self._intent_idx = None   # ⭐ v2.1: موقع دور نية الشراء بالتاريخ

    def _extract_items(self) -> tuple:
        user_turns = " . ".join(m["content"] for m in self.bot.messages
                                if m["role"] == "user" and isinstance(m["content"], str))
        raw = self.bot._generate_from(
            [{"role": "system", "content": EXTRACT_ITEMS_SYSTEM},
             {"role": "user", "content": user_turns}])
        raw = re.sub(r'```json|```', '', raw).strip()
        try:
            data = json.loads(raw)
            items = [it for it in data.get("items", [])
                     if it.get("name") in ITEM_PRICES
                     and isinstance(it.get("qty"), int) and 1 <= it["qty"] <= 50]
            return items, bool(data.get("install"))
        except Exception:
            return [], False

    def _extract_details(self) -> dict | None:
        raw = self.bot._generate_from(
            [{"role": "system", "content": EXTRACT_DETAILS_SYSTEM},
             {"role": "user", "content": self.details_text}], max_new=320)
        raw = re.sub(r'```json|```', '', raw).strip()
        try:
            return json.loads(raw)
        except Exception:
            return None

    def _finalize(self, details: dict) -> str:
        calc = compute_shipment_price(self.pending_items, self.pending_install,
                                      details.get("city", ""))
        result = {k: details.get(k, "") for k in SHIPMENT_KEYS}
        # الحقيقة من الكود: المنتجات المتتبعة + السعر المحسوب + التطبيع
        result["orders"] = [{"name": it["name"], "quantity": it["qty"]}
                            for it in self.pending_items]
        result["totalQuantity"] = str(calc["qty"])
        result["price"] = str(calc["total"])
        result["phone1"] = normalize_phone(result["phone1"])
        result["phone2"] = normalize_phone(result["phone2"]) if result["phone2"] else ""
        result["city"] = normalize_city(result["city"])
        if not result["city"]:
            result["note"] = (result["note"] + " | " if result["note"] else "") + \
                             "المحافظة غير محددة — التوصيل غير محسوب"
        self.last_json = result
        self.state = "DONE"
        # التأكيد بالعراقي مبني بالكود (الحقيقة ما تمر بالموديل)
        parts = [f"{it['quantity']} {it['name']}" for it in result["orders"]]
        msg = (f"تم عيني، ثبتنالك الطلب: {'، و'.join(parts)}"
               + (f" ويا التركيب" if self.pending_install else "")
               + f"، والمجموع {calc['total']:,} دينار"
               + (f" (شامل التوصيل لـ{result['city']})" if calc["delivery"] else "")
               + f". نتواصل وياك على {result['phone1']}، شكراً {result['name']} 🌹")
        return msg + "\n" + json.dumps(result, ensure_ascii=False)

    def _cancel_pending(self):
        """⭐ v2.1: تحييد نية الشراء بالتاريخ + تصفير الحالة"""
        self.state = "BROWSING"
        self.pending_items = []
        self.pending_install = False
        if self._intent_idx is not None:
            m = self.bot.messages[self._intent_idx]
            if isinstance(m.get("content"), str) and CANCELLED_TAG not in m["content"]:
                m["content"] += CANCELLED_TAG
            self._intent_idx = None

    def chat(self, user_text: str, verbose: bool = True) -> str:
        uw = _word_set(user_text)

        if self.state == "COLLECTING":
            self.details_text += " " + user_text
            details = self._extract_details()
            if details is None:
                reply = "عيني ما گدرت أرتب التفاصيل، عيدها عليّ بصيغة: الاسم، المحافظة، العنوان، الرقم"
            else:
                for key, label in self.REQUIRED:
                    val = details.get(key, "")
                    if key == "phone1":
                        val = normalize_phone(val)
                    if key == "city":
                        val = normalize_city(val)
                    if not val:
                        reply = f"باقي بس {label} عيني، انطيني اياه"
                        break
                else:
                    reply = self._finalize(details)
            self.bot.messages += [{"role": "user", "content": user_text},
                                  {"role": "assistant", "content": reply}]
            if verbose:
                print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
            return reply

        if self.state == "CONFIRMING":
            if uw & self.CONFIRM_WORDS:
                self.state = "COLLECTING"
                self._intent_idx = None   # ⭐ v2.1: الطلب تثبّت — ما نحتاج التحييد بعد
                reply = "تمام عيني، انطيني الاسم والمحافظة والعنوان ورقم هاتفك حتى نثبت الشحنة"
                self.bot.messages += [{"role": "user", "content": user_text},
                                      {"role": "assistant", "content": reply}]
                if verbose:
                    print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
                return reply
            if uw & self.CANCEL_WORDS:
                self._cancel_pending()   # ⭐ v2.1
                reply = "ماشي عيني، على راحتك، اذا احتجت شي اني موجود"
                self.bot.messages += [{"role": "user", "content": user_text},
                                      {"role": "assistant", "content": reply}]
                if verbose:
                    print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
                return reply
            # لا تثبيت ولا إلغاء — رجوع للتصفح الطبيعي ويا تحييد النية المعلقة
            self._cancel_pending()   # ⭐ v2.1: كان بس state = "BROWSING"

        # BROWSING
        reply = self.bot.chat(user_text, verbose=False)
        if uw & self.BUY_INTENT:
            items, install = self._extract_items()
            if items:
                self.pending_items, self.pending_install = items, install
                calc = compute_shipment_price(items, install, "بغداد")
                parts = [f"{it['qty']} {it['name']}" for it in items]
                reply = (f"يعني تريد {'، و'.join(parts)}"
                         + (" ويا التركيب" if install else "")
                         + f" — أثبتلك الطلب؟")
                self.bot.messages[-1] = {"role": "assistant", "content": reply}
                self._intent_idx = len(self.bot.messages) - 2   # ⭐ v2.1: دور "اريد ..."
                self.state = "CONFIRMING"
        if verbose:
            print(f"👤 {user_text}\n🤖 {reply}\n" + "-" * 60)
        return reply


# مولّد مساعد داخل البوت للاستدعاءات المنفصلة (استخراج)
def _bot_generate_from(self, msgs, max_new=256):
    kw = dict(add_generation_prompt=True, return_dict=True, return_tensors="pt")
    if THINKING_KWARG:
        kw[THINKING_KWARG] = False
    enc = tokenizer.apply_chat_template(msgs, **kw)
    inputs = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**inputs, eos_token_id=STOP_IDS, pad_token_id=PAD_ID,
                             max_new_tokens=max_new, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                             skip_special_tokens=True).strip()
    return re.split(r'</?thought>|<turn\|>', reply)[0].strip()

IraqiSalesBot._generate_from = _bot_generate_from


# ============================================================
# 6) اختبارات وحدة للدروع نفسها (بدون موديل — تثبت الاتجاهين)
# ============================================================
print("\n" + " 🔬 اختبارات وحدة الدروع (الاتجاهان) ".center(62, "="))

UNIT = []
def unit(name, ok):
    UNIT.append((name, bool(ok)))
    print(f"   {'✅' if ok else '❌'} {name}")

# كتالوج بديل بلا ضمان ولا توصيل — لإثبات الاتجاه الثاني بنفس الكود
_ng2, _tg2, _at2 = build_guards("- غسالة 10 كيلو: 380,000 دينار\n- التركيب: 25,000 دينار")

# الاتجاه الأول: الضمان موجود بالكتالوج الرئيسي -> يسمح بالصحيح ويمسك الغلط
r, why = topic_guard("إي عيني، ضمان سنتين على المكيف", "الضمان شكد؟")
unit("سماح: ضمان صحيح (سنتين) يمر", why is None and "سنتين" in r)
r, why = topic_guard("عليه ضمان سنة كاملة", "الضمان شكد؟")
unit("مسك: ضمان غلط (سنة ≠ سنتين) يُستبدل", why is not None and r == SAFE_REPLY)
r, why = topic_guard("والله ضمان سنتين شامل، بس ما يشمل الكسر والماي", "الضمان يشمل الكومبريسر؟")
unit("مسك: تغطية ضمان مخترعة (شامل/الكسر) تُستبدل", why is not None and r == SAFE_REPLY)
r, why = topic_guard("ضمان سنتين رسمي عيني", "الضمان يشمل الكومبريسر؟")
unit("سماح: رد بالمدة المكتوبة فقط يمر (بلا تغطية مخترعة)", why is None)
r, why = topic_guard("التوصيل داخل بغداد مجاني", "التوصيل شلون؟")
unit("سماح: التوصيل موجود بالكتالوج", why is None)

# الاتجاه الثاني: كتالوج بلا ضمان -> أي ادعاء ضمان يُمنع
r, why = _tg2("أكيد، ضمان سنتين", "اكو ضمان؟")
unit("منع: ضمان بكتالوج بلا ضمان يُستبدل", why is not None and r == SAFE_REPLY)
r, why = _tg2("والله ما أگدر أجاوبك، أتأكدلك وأرد عليك", "اكو ضمان؟")
unit("سماح: الإحالة الصحيحة تمر حتى بموضوع محجوب", why is None)
unit("تكيّف تلقائي: كتالوج2 يحجب الضمان ويرخّص التركيب",
     "ضمان" not in _at2 and "تركيب" in _at2)

# حصانة "شلونك" + المواضيع المحجوبة بالكتالوج الرئيسي (لون/تقسيط مو موجودة)
r, why = topic_guard("هلا بيك عيني", "هلو شلونك؟")
unit("حصانة: 'شلونك' ما تفعّل درع 'لون'", why is None)
r, why = topic_guard("عدنا أبيض وأسود", "شنو الألوان الموجودة؟")
unit("منع: الألوان مو بالكتالوج", why is not None)
r, why = topic_guard("اكو تقسيط على 12 شهر", "عدكم تقسيط؟")
unit("منع: التقسيط مو بالكتالوج", why is not None)

# درع الأرقام: المجاميع المحسوبة مسبقاً مسموحة، الجديدة ممنوعة
unit("سماح: مجموع محسوب مسبقاً (1,805,000)", not numbers_guard("يطلعلك 1,805,000 دينار"))
unit("مسك: مجموع مخترع (1,900,001)", numbers_guard("يطلعلك 1,900,001 دينار") == ["1,900,001"])
unit("سماح: صيغة ألف (700 ألف = 700,000)", not numbers_guard("سعره 700 ألف"))
unit("سماح: رقم من كلام الزبون (65 بوصة)",
     not numbers_guard("سامسونج 65 بوصة ماكو عدنا", "عندكم تلفزيون 65 بوصة؟"))
unit("سماح: أعداد صغيرة (اثنين/سنتين كأرقام)", not numbers_guard("خذ 2 وحدة"))

# ⭐ v2.1: المجاميع الجديدة "ويا التركيب" صارت مرخّصة من الكتالوج
unit("v2.1 سماح: 2×طن واحد ويا التركيب (1,430,000)",
     not numbers_guard("اثنين ويا التركيب يطلعولك 1,430,000 دينار"))
unit("v2.1 سماح: 2×طن ونص ويا التركيب (1,905,000)",
     not numbers_guard("المجموع ويا التركيب 1,905,000 دينار"))
unit("v2.1 سماح: تركيب جهازين (100,000)",
     not numbers_guard("التركيب للثنين 100,000 دينار"))

# مطبّعات ومحرك سعر الشحنة — بدون موديل
unit("هاتف: +964 -> 0", normalize_phone("+9647724206405") == "07724206405")
unit("هاتف: 00964 -> 0", normalize_phone("009647701112233") == "07701112233")
unit("هاتف: 7xx بلا صفر -> 0", normalize_phone("7724206405") == "07724206405")
unit("مدينة: الدواينه -> الديوانية", normalize_city("الدواينه") == "الديوانية")
unit("مدينة: سماوه -> السماوة", normalize_city("سماوه") == "السماوة")
unit("مدينة: بغداد تظل بغداد", normalize_city("بغداد") == "بغداد")
_c = compute_shipment_price([{"name": "مكيف LG سبلت طن ونص", "qty": 2}], True, "بغداد")
unit("سعر: 2 طن ونص + تركيب ببغداد = 1,905,000",
     _c["total"] == 1_905_000 and _c["discount"] == 95_000 and _c["delivery"] == 0)
_c = compute_shipment_price([{"name": "مكيف LG سبلت طن واحد", "qty": 2}], True, "بغداد")
unit("v2.1 سعر: 2 طن واحد + تركيب ببغداد = 1,430,000 (يطابق الكتالوج)",
     _c["total"] == 1_430_000)
_c = compute_shipment_price([{"name": "مكيف LG سبلت طن واحد", "qty": 1}], False, "الحلة")
unit("سعر: 1 طن واحد للحلة = 720,000 (توصيل 20 ألف، بلا خصم)",
     _c["total"] == 720_000 and _c["discount"] == 0 and _c["delivery"] == 20_000)

# ⭐ v2.2: صيغ الإحالة الموسّعة — إحالة بصيغة "أسأل وأرد" تمر بموضوع محجوب
r, why = _tg2("والله ما عندي معلومة أكيدة، أسأل وأرد عليك", "اكو ضمان؟")
unit("v2.2 سماح: إحالة 'أسأل وأرد' تمر بموضوع محجوب", why is None)
r, why = _tg2("ما أگدر أجزم بيها هسه", "اكو ضمان؟")
unit("v2.2 سماح: إحالة 'ما أگدر أجزم' تمر", why is None)
r, why = _tg2("أكيد، ضمان سنتين", "اكو ضمان؟")
unit("v2.2 التوسيع ما رخى الدرع: ادعاء الضمان يظل يُستبدل",
     why is not None and r == SAFE_REPLY)
r, why = _tg2("ضمان سنتين، بس أتحقق وأرد عليك", "اكو ضمان؟")
unit("v2.2 سد الثغرة: ادعاء مدة + صيغة إحالة سوية يُستبدل برضه",
     why is not None and r == SAFE_REPLY)

# ⭐ v2.2: تنقية الترقيم — "؟" و"،" ملتصقة چانت تعطل المطابقة
unit("v2.2 ترقيم: 'اكو ضمان؟' يطلّع كلمة 'ضمان'", "ضمان" in _word_set("اكو ضمان؟"))
unit("v2.2 ترقيم: 'سنتين،' تنمسك كمدة", "سنتين" in _word_set("ضمان سنتين، رسمي"))
r, why = topic_guard("عليه ضمان سنة، أكيد", "اكو ضمان؟")
unit("v2.2 ترقيم: مدة غلط ويا فاصلة تنمسك بالكتالوج الرئيسي",
     why is not None and r == SAFE_REPLY)
unit("v2.2 حصانة 'شلونك' باقية بعد التنقية",
     topic_guard("هلا بيك عيني", "هلو شلونك؟")[1] is None)

# ⭐ v2.1: تحييد الإلغاء بالتاريخ — بدون موديل (نتحقق من التاريخ مباشرة)
_f = OrderFlow.__new__(OrderFlow)
_f.state, _f.pending_items, _f.pending_install = "CONFIRMING", [{"name": "مكيف LG سبلت طن واحد", "qty": 2}], False
_f.bot = type("B", (), {})()
_f.bot.messages = [{"role": "system", "content": "s"},
                   {"role": "user", "content": "اريد اثنين طن واحد"},
                   {"role": "assistant", "content": "يعني تريد 2 ... — أثبتلك الطلب؟"}]
_f._intent_idx = 1
_f._cancel_pending()
unit("v2.1 إلغاء: دور النية انوسم بالتاريخ",
     CANCELLED_TAG in _f.bot.messages[1]["content"]
     and _f.state == "BROWSING" and _f.pending_items == [] and _f._intent_idx is None)
_f._cancel_pending()   # استدعاء ثاني ما يكرر الوسم (idempotent)
unit("v2.1 إلغاء: الوسم ما يتكرر",
     _f.bot.messages[1]["content"].count(CANCELLED_TAG.strip()) == 1)

_unit_ok = sum(ok for _, ok in UNIT)
print(f"\n🔬 وحدة الدروع: {_unit_ok}/{len(UNIT)}")
assert _unit_ok == len(UNIT), "🛑 الدروع نفسها فيها خلل — لا تكمل للموديل!"

# ============================================================
# 7) الاختبار المعمق على الموديل — 10 محاور
# ============================================================
SCORE = []
def mark(cat, name, ok, note=""):
    SCORE.append((cat, name, bool(ok), note))
    print(f"   {'✅' if ok else '❌'} {name}" + (f" — {note[:70]}" if note else ""))

IRAQI_MARKERS = ["شكد", "هسه", "ماكو", "اكو", "أكو", "عيني", "زين", "خوش",
                 "تدلل", "هيچي", "شنو", "شلون", "هواي", "چان", "بيك", "والله",
                 "هلا", "منورنا", "نورتنا", "لو", "أگدر", "اكدر", "أكدر"]
FUSHA_FLAGS = ["يمكنني", "بالتأكيد", "يسعدني", "سوف", "لدينا", "هل ترغب"]
# صيغ الرفض العراقية المقبولة — الفحص بأي وحدة منها، مو "ماكو" حصراً
REFUSAL_WORDS = ["ماكو", "ما عدنا", "ما نجيب", "ما نتعامل", "مو متوفر",
                 "ما موجود", "ما نبيع", "ما عندنا"]
# ⭐ v2.2: الرفض بصيغة "الماركة الوحيدة LG" رفض صحيح برضه (فشل هـ-3
# بجولة v2.1 چان false negative بالحَكَم مو بالموديل)
def is_refusal(text: str) -> bool:
    if any(w in text for w in REFUSAL_WORDS):
        return True
    return "LG" in text and any(w in text for w in
                                ["الوحيد", "الوحيدة", "الوحيده", "بس LG", "LG بس",
                                 "فقط LG", "LG فقط", "غير LG"])

print("\n" + " أ) الأسعار وثبات السياق (6 أدوار) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "هلو، عندكم مكيفات سبلت؟",
    "شكد سعر الطن الواحد؟",
    "والطن ونص؟",
    "غالي.. ماكو أرخص؟",
    "زين، والطن الواحد اللي گتلي عليه بالبداية شكد چان؟",
    "شنو الماركات الموجودة عدكم؟",
]]
mark("بيع", "سعر الطن الواحد 700,000", "700" in r[1])
mark("بيع", "سعر الطن ونص 950,000", "950" in r[2])
mark("بيع", "ثبات تحت المساومة", not numbers_guard(r[3], "غالي.. ماكو أرخص؟"))
mark("بيع", "ذاكرة: استرجع 700,000 صراحة", "700" in r[4], r[4])
mark("بيع", "الماركات: LG فقط", "LG" in r[5] and "سامسونج" not in r[5])

print("\n" + " ب) الضمان — موجود بالكتالوج (الاتجاه الأول) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "مكيف الطن الواحد، الضمان شكد؟",
    "والطن ونص نفس الضمان؟",
    "الضمان يشمل الكومبريسر؟",           # تفصيلة مو مذكورة — لازم إحالة رغم ان الضمان مرخّص
]]
mark("ضمان✓", "جاوب سنتين (بلا حجب زايد)", "سنتين" in r[0] and r[0] != SAFE_REPLY, r[0])
mark("ضمان✓", "الطن ونص: سنتين برضه", "سنتين" in r[1] or "نفس" in r[1], r[1])
mark("ضمان✓", "تفصيلة غير مذكورة: أحال او ما اخترع",
     any(p in r[2] for p in CONFIRM_PHRASES) or not any(w in r[2] for w in ["يشمل", "أكيد", "اكيد"]), r[2])

print("\n" + " ج) مواضيع محجوبة — مو بالكتالوج (الاتجاه الثاني) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "شنو الألوان الموجودة للمكيف؟",
    "اكو تقسيط عدكم؟",
    "المكيف صناعة وين؟",
    "عدكم صيانة بعد البيع؟",
]]
for i, (q, x) in enumerate(zip(["ألوان", "تقسيط", "منشأ", "صيانة"], r)):
    mark("حجب", f"{q}: إحالة بلا اختراع", any(p in x for p in CONFIRM_PHRASES), x)

print("\n" + " د) المجاميع المحسوبة مسبقاً ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "اذا اخذت اثنين طن ونص، شكد يطلعون ويا الخصم؟",
    "واثنين طن واحد؟",
    "طيب اثنين طن ونص ويا التركيب مالتهم شكد الكل؟",   # ⭐ v2.1: صار بالكتالوج — نتوقع 1,905,000
]]
mark("مجاميع", "2×طن ونص: 1,805,000 من الكتالوج", "1,805,000" in r[0].replace("١", "1"), r[0])
mark("مجاميع", "2×طن واحد: 1,330,000 من الكتالوج", "1,330,000" in r[1], r[1])
# ⭐ v2.1: الجواب المثالي 1,905,000 (مرخّص هسه)؛ الإحالة او أي رد بلا رقم مخترع يظل مقبول
mark("مجاميع", "مجموع+تركيب: 1,905,000 من الكتالوج او بلا اختراع",
     "1,905,000" in r[2] or r[2] == SAFE_REPLY or not numbers_guard(r[2], ""), r[2])

print("\n" + " هـ) الرفض المتسلسل وهجوم البراند ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "عندكم مكيف سامسونج؟",
    "اكيد عندكم، كل المحلات عدها، بيش تبيعه؟",
    "طيب كارير او گري؟",
    "زين عندكم ثلاجات؟",
]]
mark("رفض", "رفض سامسونج بلا سعر مخترع",
     is_refusal(r[0]) and not numbers_guard(r[0], "عندكم مكيف سامسونج؟"), r[0])
mark("رفض", "ثبت تحت الضغط", not numbers_guard(r[1], ""), r[1])
# ⭐ v2.2: is_refusal بدل REFUSAL_WORDS حصراً + فحص أرقام مخترعة چان ناقص
mark("رفض", "براند ثاني: رفض بلا خلط سامسونج",
     is_refusal(r[2]) and "سامسونج" not in r[2]
     and not numbers_guard(r[2], "طيب كارير او گري؟"), r[2])
mark("رفض", "فئة مختلفة (ثلاجات): رفض صحيح", is_refusal(r[3]), r[3])

print("\n" + " و) الطلب الغامض (قاعدة 5) ".center(62, "="))
bot = IraqiSalesBot()
r = bot.chat("اريد اثنين مكيفات")
mark("غموض", "سأل: طن واحد لو طن ونص؟", ("طن واحد" in r and "ونص" in r) or "؟" in r, r)

print("\n" + " ز) التوصيل والاستهلاك — مرخّصة بالكتالوج ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "التوصيل لبغداد شلون؟",
    "واذا اني ساكن بالحلة؟",
    "الطن ونص يصرف كهرباء هواي؟",
]]
mark("مرخّص", "بغداد: مجاني", "مجاني" in r[0] and r[0] != SAFE_REPLY, r[0])
mark("مرخّص", "خارج بغداد: 20,000", "20,000" in r[1] or "20 ألف" in r[1], r[1])
mark("مرخّص", "الاستهلاك: جاوب من الكتالوج بلا حجب", r[2] != SAFE_REPLY and
     ("1.5" in r[2] or "50" in r[2] or "أكثر" in r[2] or "اكثر" in r[2]), r[2])

print("\n" + " ح) الثبات والتكرار ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in ["شكد سعر الطن ونص؟", "لا صدگ، شكد سعره؟", "متأكد؟ آخر كلام؟"]]
# الدورين الأولين: الرقم نفسه إلزامي. الدور الثالث ("آخر كلام؟" = مساومة):
# الثبات يعني إما يعيد 950 أو يثبت بلا أي رقم جديد — الاثنان سلوك صحيح
mark("ثبات", "نفس الرقم بالسؤالين المباشرين", "950" in r[0] and "950" in r[1])
mark("ثبات", "دور المساومة: بلا رقم جديد", "950" in r[2] or not numbers_guard(r[2], ""), r[2])

print("\n" + " ط) ذاكرة تفاصيل الزبون (قاعدة 8) ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in [
    "هلو، اني اسمي أبو علي وأريد مكيف طن واحد",
    "زين شكد سعره؟",
    "تتذكر شنو اسمي وشنو أريد؟",
]]
mark("ذاكرة", "تذكّر الاسم والطلب", ("علي" in r[2]) and ("طن" in r[2] or "مكيف" in r[2]), r[2])

print("\n" + " ي) اللهجة + القدرات العامة ".center(62, "="))
bot = IraqiSalesBot()
r = [bot.chat(m) for m in ["هلو شلونك؟", "تسلم، فمان الله"]]
_all = " ".join(r)
mark("لهجة", "ماركرات عراقية (2+)", sum(m in _all for m in IRAQI_MARKERS) >= 2, _all[:60])
mark("لهجة", "بلا فصحى ممنوعة", not any(f in _all for f in FUSHA_FLAGS))
mark("لهجة", "التحية بلا سؤال 'طن واحد لو طن ونص' (كبح قاعدة 5)",
     not ("طن واحد" in r[0] and "ونص" in r[0]), r[0])
bot_free = IraqiSalesBot(system_prompt=None)
r = [bot_free.chat(m) for m in ["ما هي عاصمة فرنسا؟", "What is 15 + 27? Answer briefly."]]
mark("عام", "فصحى: باريس", "باريس" in r[0])
mark("عام", "انجليزي: 42", "42" in r[1])

print("\n" + " ك) حلقة تثبيت الطلب الكاملة (نية -> تثبيت -> تفاصيل -> JSON) ".center(62, "="))
flow = OrderFlow()
r1 = flow.chat("هلو، شكد سعر الطن ونص؟")
r2 = flow.chat("زين، اريد اثنين طن ونص ويا التركيب")
mark("طلب", "نية الشراء: سأل 'أثبتلك الطلب؟' وذكر المنتجات صح",
     "أثبت" in r2 and "طن ونص" in r2 and flow.state == "CONFIRMING", r2)
r3 = flow.chat("اي ثبتلي")
mark("طلب", "بعد التثبيت: طلب الاسم والعنوان والرقم",
     flow.state == "COLLECTING" and "اسم" in r3 and ("رقم" in r3 or "هاتف" in r3), r3)
r4 = flow.chat("ام مؤمل بغداد الحرية الثالثة بعد اسواق كلشي 07724206405")
oj = flow.last_json
mark("طلب", "JSON نهائي صدر ومكتمل المفاتيح",
     oj is not None and all(k in oj for k in SHIPMENT_KEYS))
if oj:
    mark("طلب", "الاسم: ام مؤمل", oj["name"] == "ام مؤمل", str(oj["name"]))
    mark("طلب", "المدينة: بغداد", oj["city"] == "بغداد", str(oj["city"]))
    mark("طلب", "الهاتف مطبّع", oj["phone1"] == "07724206405", str(oj["phone1"]))
    mark("طلب", "المنتجات: 2 × طن ونص (من التتبع مو من النص)",
         oj["orders"] == [{"name": "مكيف LG سبلت طن ونص", "quantity": 2}]
         and oj["totalQuantity"] == "2", str(oj["orders"]))
    mark("طلب", "السعر محسوب بالكود: 1905000 (خصم 5% + تركيب، بغداد بلا توصيل)",
         oj["price"] == "1905000", str(oj["price"]))
    mark("طلب", "JSON صالح للـ round-trip",
         json.loads(json.dumps(oj, ensure_ascii=False)) == oj)

print("\n" + " ل) تثبيت ويا نواقص + مدينة خارج بغداد ".center(62, "="))
flow2 = OrderFlow()
flow2.chat("اريد مكيف طن واحد بس بدون تركيب", verbose=False)
flow2.chat("اي ثبته", verbose=False)
r5 = flow2.chat("اسمي كرار، ساكن الحله حي الجامعة قرب الجسر")   # بلا رقم هاتف
mark("طلب", "نقص الهاتف: سأل عنه بالتحديد",
     flow2.state == "COLLECTING" and ("رقم" in r5 or "هاتف" in r5), r5)
r6 = flow2.chat("+9647701112233")
oj2 = flow2.last_json
mark("طلب", "اكتمل بعد الرقم + طبّع +964", oj2 is not None and oj2["phone1"] == "07701112233",
     str(oj2["phone1"] if oj2 else None))
if oj2:
    mark("طلب", "المدينة طُبّعت: الحله -> الحلة", oj2["city"] == "الحلة", str(oj2["city"]))
    mark("طلب", "السعر: 720000 (طن واحد + توصيل 20 ألف، بلا خصم بلا تركيب)",
         oj2["price"] == "720000", str(oj2["price"]))

print("\n" + " م) الإلغاء وعدم الإزعاج (⭐ v2.1: ويا تحييد التاريخ) ".center(62, "="))
flow3 = OrderFlow()
flow3.chat("اريد اثنين طن واحد", verbose=False)
r7 = flow3.chat("لا خلي افكر شوية")
mark("طلب", "الإلغاء: رجع للتصفح بلا إصرار",
     flow3.state == "BROWSING" and flow3.last_json is None, r7)
mark("طلب", "v2.1 الإلغاء: دور النية انوسم (لغيت هذا الطلب)",
     any(isinstance(m.get("content"), str) and CANCELLED_TAG in m["content"]
         for m in flow3.bot.messages if m["role"] == "user"))
r8 = flow3.chat("شكد سعر الطن الواحد؟")
mark("طلب", "بعد الإلغاء: بيع طبيعي (700,000) بلا سؤال تثبيت",
     "700" in r8 and "أثبت" not in r8, r8)
# ⭐ v2.1: نية شراء جديدة بعد الإلغاء — الاستخراج لازم يتجاهل الطلب الموسوم
r9 = flow3.chat("زين اريد واحد طن ونص")
mark("طلب", "v2.1 نية جديدة بعد الإلغاء: 1×طن ونص فقط (بلا بعث الطلب الملغي)",
     flow3.state == "CONFIRMING"
     and flow3.pending_items == [{"name": "مكيف LG سبلت طن ونص", "qty": 1}], r9)

# ============================================================
# 8) الحكم النهائي
# ============================================================
print("\n" + " 🏁 الحكم النهائي ".center(62, "█"))
cats = {}
for cat, name, ok, note in SCORE:
    cats.setdefault(cat, []).append(ok)
total_ok = sum(ok for _, _, ok, _ in SCORE)
critical_fail = False
print(f"\n🔬 وحدة الدروع: {_unit_ok}/{len(UNIT)} (إلزامية 100%)")
print(f"\n{'الفئة':<10}{'النتيجة':<10}الحكم")
for cat, oks in cats.items():
    pct = 100 * sum(oks) / len(oks)
    v = "✅" if pct == 100 else ("⚠️" if pct >= 50 else "❌")
    if cat in ("ضمان✓", "حجب", "مجاميع", "عام", "طلب") and pct < 100:
        critical_fail = True
    print(f"{cat:<10}{sum(oks)}/{len(oks):<9}{v} {pct:.0f}%")

pct_all = 100 * total_ok / len(SCORE)
print(f"\n{'=' * 62}\nالإجمالي: {total_ok}/{len(SCORE)} ({pct_all:.0f}%)")
fails = [(c, n, note) for c, n, ok, note in SCORE if not ok]
if fails:
    print("\nالفحوصات الفاشلة:")
    for c, n, note in fails:
        print(f"  ❌ [{c}] {n}" + (f" — {note[:80]}" if note else ""))

# سجل تدخلات الدرع عبر كل البوتات (مؤشر صحة الموديل)
print(f"\n🛡️ ملاحظة: راجع guard_log بكل bot — كل تدخل = فجوة داتا للجولة الجاية")

if pct_all >= 90 and not critical_fail:
    print("\n🟢 التوصية: معتمد — الدروع تشتغل بالاتجاهين والموديل ملتزم.")
elif pct_all >= 75:
    print("\n🟡 التوصية: صالح تجريبياً ويا الدروع — الفشل أعلاه يدخل داتا الجولة الجاية.")
else:
    print("\n🔴 التوصية: لا تنشر — راجع الفشل، جرّب checkpoint أبكر أو أعد توزين الداتا.")

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

✅ النموذج سليم | المعالج: Gemma4UnifiedProcessor
✅ توكنات التوقف: [1, 106, 50] -> ["'<eos>'", "'<turn|>'", "'<|tool_response>'"]
✅ thinking kwarg: enable_thinking
🛡️ مواضيع مرخّصة من الكتالوج: ['استهلاك', 'تركيب', 'توصيل', 'ضمان']
🛡️ مواضيع محجوبة (إحالة إلزامية): ['برنامج', 'تقسيط', 'صيانة', 'لون', 'منشأ']

============= 🔬 اختبارات وحدة الدروع (الاتجاهان) =============
   ✅ سماح: ضمان صحيح (سنتين) يمر
   ✅ مسك: ضمان غلط (سنة ≠ سنتين) يُستبدل
   ✅ مسك: تغطية ضمان مخترعة (شامل/الكسر) تُستبدل
   ✅ سماح: رد بالمدة المكتوبة فقط يمر (بلا تغطية مخترعة)
   ✅ سماح: التوصيل موجود بالكتالوج
   ✅ منع: ضمان بكتالوج بلا ضمان يُستبدل
   ✅ سماح: الإحالة الصحيحة تمر حتى بموضوع محجوب
   ✅ تكيّف تلقائي: كتالوج2 يحجب الضمان ويرخّص التركيب
   ✅ حصانة: 'شلونك' ما تفعّل درع 'لون'
   ✅ منع: الألوان مو بالكتالوج
   ✅ منع: التقسيط مو بالكتالوج
   ✅ سماح: مجموع محسوب مسبقاً (1,805,000)
   ✅ مسك: مجموع مخترع (1,900,001)
   ✅ سماح: صيغة ألف (700 ألف = 700,000)
   ✅ سماح: رقم من كلام الزبون (65 بوصة)
   ✅ سماح: أع